# Experiment 2: one-dimensional triple well

This notebook studies a three-mode Gibbs target and the effect of jump-graph
design.  An adjacent shell connects neighboring modes, while an overlong shell
connects the outer modes more directly and can underrepresent the middle mode.
The notebook compares local baselines, corrected adjacent and overlong jumps,
and a raw overlong jump control using direct mode masses, mode-TV, weak errors,
and transition diagnostics.


## Scientific workflow and outputs

The notebook computes the exact one-dimensional target by quadrature, the
three mode regions and masses, adjacent and overlong jump supports, Levy-score
corrections, sampler diagnostics, and output provenance.  Tables and PDF
figures are written under the four-experiment manuscript release tree.


In [ ]:
import os, math, time, warnings, sys
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Callable

import numpy as np
import pandas as pd
import matplotlib
from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy import sparse
from scipy.linalg import eigh
from scipy.special import logsumexp, erf, erfcx
from scipy.stats import wasserstein_distance
try:
    from scipy.ndimage import gaussian_filter
except Exception:
    gaussian_filter = None

try:
    from joblib import Parallel, delayed
    HAS_JOBLIB = True
except Exception:
    HAS_JOBLIB = False

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 350,
    "font.size": 9.5,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

PROFILE = os.environ.get("LEVY_PROFILE", "paperlite").lower()
assert PROFILE in {"smoke", "paperlite", "paper"}
print("LEVY_PROFILE =", PROFILE)

GLOBAL_SEED = 20260430

# NumPy 1.x / 2.x compatible trapezoidal rule helper.
_trapz = getattr(np, "trapezoid", np.trapz)


def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path

def find_project_root(start=None):
    path = Path.cwd() if start is None else Path(start).resolve()
    while path.name != "levy-score-sampling-project" and path.parent != path:
        path = path.parent
    if path.name != "levy-score-sampling-project":
        raise RuntimeError("Could not locate levy-score-sampling-project from current working directory")
    return path

PROJECT_ROOT = find_project_root()
RELEASE_ROOT = ensure_dir(PROJECT_ROOT / "manuscript_clean_active" / "numerics" / "four_experiment_release")
RELEASE_TABLE_DIR = ensure_dir(RELEASE_ROOT / "tables")
RELEASE_LOG_DIR = ensure_dir(RELEASE_ROOT / "logs")
RELEASE_FIG_ROOT = ensure_dir(PROJECT_ROOT / "manuscript_clean_active" / "figures" / "four_experiment_release")
MAIN_FIG_DIR = ensure_dir(RELEASE_FIG_ROOT / "main_candidates")
APPENDIX_FIG_DIR = ensure_dir(RELEASE_FIG_ROOT / "appendix_candidates")
DIAGNOSTIC_FIG_DIR = ensure_dir(RELEASE_FIG_ROOT / "diagnostics")
MANUSCRIPT_FIG_DIR = DIAGNOSTIC_FIG_DIR
CANDIDATE_RESULT_DIR = RELEASE_ROOT
COMMON_DIR = RELEASE_ROOT / "common"
if str(COMMON_DIR) not in sys.path:
    sys.path.insert(0, str(COMMON_DIR))
from levy.one_d_density import (
# Canonical plotting and spectral-reference helpers are imported below.
    DensityDiagnosticConfig,
    binned_gaussian_kde_on_grid,
    binned_gaussian_kde_on_grid_with_diagnostics,
    central_interval_from_cdf,
    density_errors_on_grid,
    density_metric_bundle,
    estimate_kde_bias_floor,
    kde_histogram_bin_mass_l1,
    normalize_mixture_weights,
    summarize_density_diagnostics,
    target_local_scott_bandwidth,
)
def savefig(fig, outdir, name):
    outdir = ensure_dir(outdir)
    try:
        fig.tight_layout()
    except Exception:
        pass
    stem = str(name)
    if not stem.startswith(FIG_PREFIX):
        stem = FIG_PREFIX + stem
    pdf = outdir / f"{stem}.pdf"
    fig.savefig(pdf, bbox_inches="tight")
    print("saved", pdf)
    return pdf
def trapz_weights(x):
    x = np.asarray(x)
    w = np.empty_like(x, dtype=float)
    dx = np.diff(x)
    w[1:-1] = 0.5*(dx[:-1] + dx[1:])
    w[0] = 0.5*dx[0]
    w[-1] = 0.5*dx[-1]
    return w

def normalize_pdf_grid(x, logp):
    w = trapz_weights(x)
    lp = logp - np.max(logp)
    raw = np.exp(lp)
    Z = np.sum(raw*w)
    p = raw/Z
    return p, w, Z

def empirical_pdf_1d(samples, grid, clip_range=None):
    if clip_range is None:
        clip_range = (grid[0], grid[-1])
    hist, edges = np.histogram(samples, bins=len(grid)-1, range=clip_range, density=True)
    centers = 0.5*(edges[:-1]+edges[1:])
    return centers, hist

def empirical_cdf_on_grid(samples, grid):
    s = np.sort(samples)
    return np.searchsorted(s, grid, side='right') / max(1, len(s))

def cdf_from_pdf(x, p):
    w = trapz_weights(x)
    c = np.cumsum(p*w)
    c = c / c[-1]
    return c

def mmd_rbf_1d(x, y, bandwidth=0.7, max_points=1200, rng=None, block_size=128):
    rng = np.random.default_rng(123) if rng is None else rng
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    if len(x) > max_points:
        x = rng.choice(x, size=max_points, replace=False)
    if len(y) > max_points:
        y = rng.choice(y, size=max_points, replace=False)

    def kernel_mean(a, b):
        total = 0.0
        count = 0
        for i in range(0, len(a), block_size):
            aa = a[i:i+block_size, None]
            for j in range(0, len(b), block_size):
                bb = b[None, j:j+block_size]
                block = np.exp(-0.5*((aa-bb)/bandwidth)**2)
                total += float(block.sum())
                count += block.size
        return total / max(count, 1)

    return float(kernel_mean(x, x) + kernel_mean(y, y) - 2.0*kernel_mean(x, y))

def weighted_choice_indices(rng, probs, size):
    probs = np.asarray(probs, dtype=float)
    probs = probs / probs.sum()
    return rng.choice(len(probs), size=size, p=probs)

def w1_empirical_to_target_cdf(samples, target_grid, target_cdf):
    """Approximate 1D W1 between empirical samples and the target CDF.

    This avoids an additional Monte Carlo reference-subsampling noise source.
    The target CDF is linearly interpolated on target_grid and extended by
    constants outside the displayed/grid interval.
    """
    samples = np.asarray(samples, dtype=float)
    grid = np.asarray(target_grid, dtype=float)
    cdf = np.asarray(target_cdf, dtype=float)
    if samples.size == 0:
        return np.nan
    pts = np.unique(np.concatenate([grid, samples[np.isfinite(samples)]]))
    if pts.size < 2:
        return 0.0
    F_emp = np.searchsorted(np.sort(samples), pts, side="right") / max(1, samples.size)
    F_tar = np.interp(pts, grid, cdf, left=0.0, right=1.0)
    return float(_trapz(np.abs(F_emp - F_tar), pts))

plt.rcParams.update({
    "figure.dpi": 135,
    "savefig.dpi": 300,
    "font.size": 9.5,
    "axes.titlesize": 10.5,
    "axes.labelsize": 9.5,
    "legend.fontsize": 8,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "grid.alpha": 0.22,
    "lines.linewidth": 1.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def panel_label(ax, label, x=-0.12, y=1.06):
    ax.text(x, y, label, transform=ax.transAxes, fontsize=12, fontweight="bold",
            va="top", ha="left")

def clean_axes(ax, grid=True):
    if grid:
        ax.grid(alpha=0.22, lw=0.55)
    ax.tick_params(length=3, width=0.7)
    return ax



from levy.plot_style import (
    METHOD_STYLES as CANONICAL_METHOD_STYLES,
    apply_plot_style,
    method_color as canonical_method_color,
    method_marker as canonical_method_marker,
)
from levy.spectral_references import add_spectral_reference_lines, write_reference_registry
apply_plot_style(plt)


In [ ]:
# -------------------------------
# Plotting and statistical helpers
# -------------------------------
METHOD_ORDER = [
    "Langevin", "Kinetic-Langevin", "LSC-CP", "CP",
    "LSC-adjacent", "LSC-overlong", "CP-overlong",
    "LSC-connected", "LSC-disconnected", "LSC-wrong",
]
METHOD_COLORS = {name: style.color for name, style in CANONICAL_METHOD_STYLES.items()}
METHOD_MARKERS = {name: style.marker for name, style in CANONICAL_METHOD_STYLES.items()}
FIGSIZE_ROW = (13.2, 4.2)
FIGSIZE_GRID = (15.2, 8.6)
FIGSIZE_DIAGNOSTIC = (14.5, 4.2)

def method_color(method):
    return canonical_method_color(str(method))

def method_marker(method):
    return canonical_method_marker(str(method))

def mean_se_by_time(df, method, metric):
    sub = df[df.method == method].groupby("time")[metric].agg(["mean", "std", "count"])
    sub["std"] = sub["std"].fillna(0.0)
    sub["se"] = sub["std"] / np.sqrt(np.maximum(sub["count"], 1))
    return sub

def plot_metric_with_ci(ax, df, method, metric, *, logy=True, marker=None, lw=1.5, label=None, color=None, alpha_fill=0.12):
    sub = mean_se_by_time(df, method, metric)
    if len(sub) == 0:
        return None
    y = np.asarray(sub["mean"], dtype=float)
    lo = np.asarray(sub["mean"] - 2*sub["se"], dtype=float)
    hi = np.asarray(sub["mean"] + 2*sub["se"], dtype=float)
    c = color or method_color(method)
    mk = marker or method_marker(method)
    if logy:
        floor = max(1e-12, 1e-6*np.nanmax(np.abs(y)) if np.nanmax(np.abs(y)) > 0 else 1e-12)
        y = np.maximum(y, floor)
        lo = np.maximum(lo, floor)
        hi = np.maximum(hi, floor)
        ax.set_yscale("log")
    ax.plot(sub.index, y, marker=mk, ms=3, lw=lw, color=c, label=label or method)
    ax.fill_between(sub.index, lo, hi, alpha=alpha_fill, color=c, linewidth=0)
    return sub


SPECTRAL_REFERENCE_RECORDS = []
SPECTRAL_REFERENCE_EXPERIMENT = "triple-well"


SPECTRAL_REFERENCE_RECORDS = []
SPECTRAL_REFERENCE_EXPERIMENT = "triple-well"


SPECTRAL_REFERENCE_RECORDS = []
SPECTRAL_REFERENCE_EXPERIMENT = "triple-well"

def add_rate_reference_lines(ax, df, metric, method, *, form_rate=None, abscissa_rate=None, tmax=None, logy=True):
    return add_spectral_reference_lines(
        ax,
        df,
        metric,
        method,
        form_rate=form_rate,
        abscissa_rate=abscissa_rate,
        records=SPECTRAL_REFERENCE_RECORDS,
        experiment=SPECTRAL_REFERENCE_EXPERIMENT,
        figure=f"{SPECTRAL_REFERENCE_EXPERIMENT}:{metric}:time-decay",
        tmax=tmax,
        logy=logy,
    )

def fit_observed_decay_rates(df, methods, metrics, *, t_min_frac=0.25, t_max_frac=0.85, min_points=4, floor_rel=1e-5):
    rows = []
    t0 = float(df.time.min())
    t1 = float(df.time.max())
    lo_t = t0 + t_min_frac*(t1-t0)
    hi_t = t0 + t_max_frac*(t1-t0)
    for method in methods:
        for metric in metrics:
            sub = mean_se_by_time(df, method, metric)
            if len(sub) == 0:
                rows.append(dict(method=method, metric=metric, r_obs=np.nan, r_obs_se=np.nan, n_fit=0, t_min=np.nan, t_max=np.nan))
                continue
            times = np.asarray(sub.index, dtype=float)
            y = np.asarray(sub["mean"], dtype=float)
            y_floor = max(1e-14, floor_rel*np.nanmax(np.abs(y)) if np.nanmax(np.abs(y)) > 0 else 1e-14)
            mask = (times >= lo_t) & (times <= hi_t) & np.isfinite(y) & (y > y_floor)
            if np.sum(mask) < min_points:
                mask = (times > t0) & np.isfinite(y) & (y > y_floor)
            if np.sum(mask) < min_points:
                rows.append(dict(method=method, metric=metric, r_obs=np.nan, r_obs_se=np.nan, n_fit=int(np.sum(mask)), t_min=np.nan, t_max=np.nan))
                continue
            x = times[mask]
            z = np.log(np.maximum(y[mask], y_floor))
            slope, intercept = np.polyfit(x, z, 1)
            resid = z - (slope*x + intercept)
            sxx = np.sum((x - x.mean())**2)
            sigma2 = np.sum(resid**2) / max(1, len(x)-2)
            slope_se = np.sqrt(sigma2 / max(sxx, 1e-300))
            rows.append(dict(method=method, metric=metric, r_obs=float(-slope), r_obs_se=float(slope_se),
                             n_fit=int(len(x)), t_min=float(x.min()), t_max=float(x.max())))
    return pd.DataFrame(rows)



def fit_seed_level_decay_rates(df, methods, metrics, *, t_min_frac=0.25, t_max_frac=0.85, min_points=4, floor_rel=1e-5):
    """Fit log-linear decay separately for each seed, then summarize across seeds.

    The ordinary curve-level fit treats time points as independent.  This seed-level
    summary is a more honest uncertainty diagnostic for finite-seed Monte Carlo curves.
    """
    seed_rows = []
    summary_rows = []
    t0 = float(df.time.min())
    t1 = float(df.time.max())
    lo_t = t0 + t_min_frac*(t1-t0)
    hi_t = t0 + t_max_frac*(t1-t0)
    for method in methods:
        method_df = df[df.method == method]
        for metric in metrics:
            per_rates = []
            for seed, sub_seed in method_df.groupby("seed"):
                sub_seed = sub_seed.sort_values("time")
                times = np.asarray(sub_seed["time"], dtype=float)
                y = np.asarray(sub_seed[metric], dtype=float)
                if len(times) == 0 or not np.any(np.isfinite(y)):
                    seed_rows.append(dict(method=method, metric=metric, seed=seed, r_obs_seed=np.nan,
                                          n_fit=0, t_min=np.nan, t_max=np.nan))
                    continue
                y_floor = max(1e-14, floor_rel*np.nanmax(np.abs(y)) if np.nanmax(np.abs(y)) > 0 else 1e-14)
                mask = (times >= lo_t) & (times <= hi_t) & np.isfinite(y) & (y > y_floor)
                if np.sum(mask) < min_points:
                    mask = (times > t0) & np.isfinite(y) & (y > y_floor)
                if np.sum(mask) < min_points:
                    seed_rows.append(dict(method=method, metric=metric, seed=seed, r_obs_seed=np.nan,
                                          n_fit=int(np.sum(mask)), t_min=np.nan, t_max=np.nan))
                    continue
                x = times[mask]
                z = np.log(np.maximum(y[mask], y_floor))
                slope, intercept = np.polyfit(x, z, 1)
                r_seed = float(-slope)
                per_rates.append(r_seed)
                seed_rows.append(dict(method=method, metric=metric, seed=seed, r_obs_seed=r_seed,
                                      n_fit=int(len(x)), t_min=float(x.min()), t_max=float(x.max())))
            rates = np.asarray([r for r in per_rates if np.isfinite(r)], dtype=float)
            if rates.size == 0:
                summary_rows.append(dict(method=method, metric=metric, r_obs=np.nan, r_obs_se=np.nan,
                                         r_obs_seed_std=np.nan, n_seed_fit=0, t_min=lo_t, t_max=hi_t))
            else:
                seed_std = float(np.std(rates, ddof=1)) if rates.size > 1 else np.nan
                seed_se = float(seed_std/np.sqrt(rates.size)) if rates.size > 1 else np.nan
                summary_rows.append(dict(method=method, metric=metric, r_obs=float(np.mean(rates)),
                                         r_obs_se=seed_se, r_obs_seed_std=seed_std,
                                         n_seed_fit=int(rates.size), t_min=lo_t, t_max=hi_t))
    return pd.DataFrame(summary_rows), pd.DataFrame(seed_rows)

def make_rng_seed(*items):
    ss = np.random.SeedSequence([int(abs(x)) % (2**32-1) for x in items])
    return int(ss.generate_state(1)[0])


# Canonical method styles are supplied by levy.plot_style.


## Numerical protocol

The numerical protocol is organized around the Dirichlet-form gap.  In this notebook it is used as a conservative $$L^2(\mu)$$ reference rate for the symmetric diagonal form.  It is not claimed to be the exact finite-time rate of Wasserstein, CDF, mode-TV, weak-error, or other empirical metrics.

The generator abscissa is a finite-dimensional diagnostic for the discretized nonreversible generator.  It is reported together with row-sum error, stationary residual, stationary bias, negative-mass diagnostics for the stationary vector, and domain/grid sensitivity.  It should be read as a numerical diagnostic, not as a theorem-level continuous-generator rate.

The observed rate is a finite-time, metric-dependent Monte Carlo summary from a log-linear fit.  Its reported standard error is a regression diagnostic, not a full Monte Carlo confidence interval.  The shaded metric curves remain the main seed-level uncertainty visualization.

Reference lines on empirical curves are visual comparisons only: they start from the same empirical initial value as the plotted metric and should not be read as rigorous upper or lower bounds for that empirical metric.


### Interpretation guardrails

The numerical protocol is organized around the Dirichlet-form gap, which is used as a conservative $$L^2(\mu)$$ reference rate for the diagonal form.  It is not asserted to be the exact finite-time decay rate of Wasserstein, CDF, mode-TV, weak-error, or any prescribed initial cloud.

The finite-dimensional generator abscissa is a diagnostic for the discretized nonreversible pseudo-generator.  It should be interpreted only together with the stationary residual, stationary-bias, negative-mass, grid, and domain diagnostics.

The fitted observed rate is metric-dependent and finite-time.  Its standard error in the compact comparison table is computed across seed-level fitted slopes when available; it is not a theorem-level confidence interval for the Markov semigroup.


In [ ]:
# -------------------------------
# 1D jump law, Levy score, gap
# -------------------------------
@dataclass
class ShellJump1D:
    a: float
    h: float
    lam: float
    quad_r: int = 7
    name: str = "shell"

    def atoms(self):
        # Gauss--Legendre quadrature for the two shell intervals.
        z, w = np.polynomial.legendre.leggauss(self.quad_r)
        pos = self.a + self.h * z
        neg = -self.a + self.h * z
        # Each interval has mass 1/2; GL weights sum to 2.
        rs = np.concatenate([pos, neg])
        ws = np.concatenate([0.25*w, 0.25*w])
        ws = ws / ws.sum()
        return rs.astype(float), ws.astype(float)

    def density_on_grid(self, rgrid):
        r = np.asarray(rgrid)
        return ((np.abs(np.abs(r) - self.a) <= self.h).astype(float) / (4*self.h))

    def sample_jumps(self, rng, size):
        # Exact shell sampling, not quadrature sampling.
        signs = rng.choice(np.array([-1.0, 1.0]), size=size)
        mags = rng.uniform(self.a-self.h, self.a+self.h, size=size)
        return signs*mags

def levy_score_1d_grid(xgrid, logp_fn, jump: ShellJump1D, theta_n=24, log_clip=None, score_clip=None):
    """Evaluate the 1D Levy-score correction on a grid.

    log_clip=None applies only the floating-point overflow guard.  Passing a
    smaller log_clip is score-changing truncation and is recorded explicitly.
    """
    x = np.asarray(xgrid)
    zt, wt = np.polynomial.legendre.leggauss(theta_n)
    theta = 0.5*(zt+1.0); wt=0.5*wt
    rs, wr = jump.atoms()
    lp_x = logp_fn(x)
    S = np.zeros_like(x)
    score_changing_clip_count = 0
    overflow_guard_clip_count = 0
    total_count = 0
    max_log_ratio = 0.0
    float_log_clip = float(np.log(np.finfo(float).max) - 2.0)
    effective_log_clip = float_log_clip if log_clip is None else min(float(log_clip), float_log_clip)
    for r, pr in zip(rs, wr):
        acc = np.zeros_like(x)
        for th, wth in zip(theta, wt):
            dlog = logp_fn(x - th*r) - lp_x
            max_log_ratio = max(max_log_ratio, float(np.max(np.abs(dlog))))
            if log_clip is not None:
                score_changing_clip_count += int(np.sum(np.abs(dlog) > effective_log_clip))
            overflow_guard_clip_count += int(np.sum(np.abs(dlog) > float_log_clip))
            total_count += dlog.size
            ratio = np.exp(np.clip(dlog, -effective_log_clip, effective_log_clip))
            acc += wth * r * ratio
        S += pr*acc
    rawS = -jump.lam*S
    if score_clip is None:
        S = rawS
        clip_fraction = 0.0
    else:
        S = np.clip(rawS, -score_clip, score_clip)
        clip_fraction = float(np.mean(np.abs(rawS) > score_clip))
    logratio_clip_fraction = score_changing_clip_count/max(1,total_count)
    overflow_fraction = overflow_guard_clip_count/max(1,total_count)
    return S, {"score_clip_fraction": clip_fraction,
               "logratio_clip_fraction": logratio_clip_fraction,
               "score_changing_logratio_clip_fraction": logratio_clip_fraction,
               "overflow_guard_logratio_clip_fraction": overflow_fraction,
               "effective_log_clip": float(effective_log_clip),
               "max_score_abs": float(np.max(np.abs(rawS))) if rawS.size else 0.0,
               "max_log_ratio": float(max_log_ratio),
               "score_clipping_warning": bool(clip_fraction > 1e-3),
               "logratio_clipping_warning": bool(logratio_clip_fraction > 1e-3)}

def interp_shift_matrix_1d(x, r):
    """
    Sparse matrix T_r such that (T_r f)_i approximates f(x_i + r)
    by linear interpolation. Values outside the computational interval
    are clamped to the nearest boundary. The interval is chosen so that
    boundary mass is negligible; boundary diagnostics are reported.
    """
    x = np.asarray(x)
    n = len(x)
    y = x + r
    idx = np.searchsorted(x, y) - 1
    idx = np.clip(idx, 0, n-2)
    x0 = x[idx]; x1 = x[idx+1]
    t = np.clip((y-x0)/(x1-x0), 0.0, 1.0)
    rows = np.repeat(np.arange(n), 2)
    cols = np.ravel(np.column_stack([idx, idx+1]))
    data = np.ravel(np.column_stack([1-t, t]))
    return sparse.csr_matrix((data, (rows, cols)), shape=(n,n))

def full_gap_1d_form(logp_fn, eps, x_min, x_max, N=360, jump: Optional[ShellJump1D]=None, tail_trim=1e-10):
    """
    Compute full 1D form gap using H = M^{-1/2} K M^{-1/2}.
    Returns lambda_2, diagnostics, eigenvalues.
    """
    x_full = np.linspace(x_min, x_max, N)
    p_full, dxw_full, _ = normalize_pdf_grid(x_full, logp_fn(x_full))
    mu_full = p_full * dxw_full
    cdf = np.cumsum(mu_full); cdf /= cdf[-1]
    keep = (cdf >= tail_trim/2) & (cdf <= 1-tail_trim/2)
    if keep.sum() < 0.8*N:
        keep = mu_full > (np.max(mu_full)*1e-13)
    if keep.sum() < 20:
        keep = np.ones_like(mu_full, dtype=bool)
    x = x_full[keep]
    p, dxw, _ = normalize_pdf_grid(x, logp_fn(x))
    mu = p*dxw
    n = len(x)
    h = np.diff(x)
    K = sparse.csr_matrix((n,n), dtype=float)

    # Diffusion stiffness: eps int |f'|^2 p dx.
    diag = np.zeros(n); off = np.zeros(n-1)
    pmid = 0.5*(p[:-1]+p[1:])
    c = eps * pmid / h
    diag[:-1] += c; diag[1:] += c; off -= c
    Kdiff = sparse.diags([off, diag, off], offsets=[-1,0,1], format="csr")
    K = K + Kdiff

    boundary_jump_mass = 0.0
    if jump is not None and jump.lam > 0:
        rs, wr = jump.atoms()
        I = sparse.identity(n, format="csr")
        Mdiag = sparse.diags(mu, format="csr")
        for r, pr in zip(rs, wr):
            y = x + r
            boundary_jump_mass += float(pr * np.sum(mu[(y < x[0]) | (y > x[-1])]))
            T = interp_shift_matrix_1d(x, r)
            D = T - I
            K = K + (0.5*jump.lam*pr) * (D.T @ Mdiag @ D)

    invsqrt = 1/np.sqrt(np.maximum(mu, 1e-300))
    H = sparse.diags(invsqrt) @ K @ sparse.diags(invsqrt)
    H = 0.5*(H + H.T)
    Hd = H.toarray()
    evals, evecs = eigh(Hd)
    evals = np.maximum(evals, 0.0)
    gap = float(evals[1])
    z = evecs[:,1]
    rq = float(z @ (Hd @ z) / (z @ z))
    resid = float(np.linalg.norm(Hd @ z - gap*z) / max(1.0, abs(gap)))
    diag = {
        "N_input": N, "N_active": n,
        "x_min_active": float(x[0]), "x_max_active": float(x[-1]),
        "retained_mass_before_renorm": float(mu_full[keep].sum()),
        "boundary_jump_mass_proxy": boundary_jump_mass,
        "gap": gap, "rayleigh": rq, "residual": resid,
        "lambda0": float(evals[0]), "lambda2": gap, "lambda3": float(evals[2]) if len(evals)>2 else np.nan,
        "lambda4": float(evals[3]) if len(evals)>3 else np.nan,
        "lambda5": float(evals[4]) if len(evals)>4 else np.nan,
        "ratio32": float(evals[2]/evals[1]) if len(evals)>2 and evals[1] > 0 else np.nan,
        "ratio43": float(evals[3]/evals[2]) if len(evals)>3 and evals[2] > 0 else np.nan,
        "ratio54": float(evals[4]/evals[3]) if len(evals)>4 and evals[3] > 0 else np.nan,
    }
    return gap, diag, evals[:8], x, mu

def run_gap_convergence(logp_fn, eps, interval, Ns, jump_map):
    rows = []
    for name, jump in jump_map.items():
        for N in Ns:
            t0=time.time()
            gap, diag, evals, _, _ = full_gap_1d_form(logp_fn, eps, interval[0], interval[1], N=N, jump=jump)
            diag.update({"method": name, "elapsed_sec": time.time()-t0})
            rows.append(diag)
    return pd.DataFrame(rows)

def choose_main_gaps(gap_df):
    # Robust: no float filtering, no empty slices. Choose largest N available for each method.
    out = {}
    for method, sub in gap_df.groupby("method"):
        sub = sub.dropna(subset=["gap"]).sort_values(["N_input"])
        if len(sub)==0:
            out[method] = np.nan
        else:
            out[method] = float(sub.iloc[-1]["gap"])
    return out

def make_initial_cloud_1d(rng, center, scale, n, clip):
    x0 = center + scale*rng.standard_normal(n)
    return np.clip(x0, clip[0], clip[1])

def apply_compound_poisson_jumps_1d(x, jump, rng, dt):
    """Vectorized compound-Poisson jump accumulation for 1D particles.

    This is algebraically identical to looping over particles and summing their
    independent jumps, but avoids the Python-level per-particle loop.
    """
    nj = rng.poisson(jump.lam * dt, size=x.shape[0])
    total = int(nj.sum())
    if total <= 0:
        return x, nj
    owners = np.repeat(np.arange(x.shape[0]), nj)
    jumps = jump.sample_jumps(rng, total)
    incr = np.bincount(owners, weights=jumps, minlength=x.shape[0])
    return x + incr, nj

def safety_clip_1d(x, method, clip):
    """Safety clip for explicit discretizations.

    The returned fraction is a diagnostic of whether the computational
    domain is wide enough. In reported runs it should be essentially zero.
    """
    y = np.clip(x, *clip)
    return y, float(np.mean(y != x))


In [ ]:
OUTDIR = ensure_dir(CANDIDATE_RESULT_DIR / "threewell")
FIGDIR = MANUSCRIPT_FIG_DIR
FIG_PREFIX = "threewell_"
TABDIR = ensure_dir(RELEASE_TABLE_DIR / "02_triple_well")

if PROFILE == "smoke":
    CFG = dict(n_particles=40, n_steps=20, dt=0.004, n_seeds=1, record_every=5,
               gap_Ns=[30], gap_interval=(-6.0,6.0), grid_N=120, theta_n=4, quad_r=3, kinetic_gamma=1.3, kinetic_mass=0.02, n_jobs=1)
elif PROFILE == "paperlite":
    CFG = dict(n_particles=2400, n_steps=12000, dt=0.00035, n_seeds=4, record_every=42,
               gap_Ns=[180,280,420], gap_interval=(-12.0,12.0), grid_N=1200, theta_n=18, quad_r=7, kinetic_gamma=1.3, kinetic_mass=0.02, n_jobs=1)
else:
    CFG = dict(n_particles=10000, n_steps=5600, dt=0.0025, n_seeds=8, record_every=35,
               gap_Ns=[240,380,560,760], gap_interval=(-12.5,12.5), grid_N=1800, theta_n=30, quad_r=9, kinetic_gamma=1.3, kinetic_mass=0.02, n_jobs=min(6, os.cpu_count() or 1))

# Target is a smooth Boltzmann density represented as a separated Gaussian mixture.
# This gives exact log-density and gradient without hiding a polynomial tuning step.
eps = 0.08
x_clip = (-12.0,12.0)
modes = np.array([-3.0, 0.0, 3.0])
raw_weights = np.array([0.25, 0.45, 0.35])
weights = normalize_mixture_weights(raw_weights)
weight_normalization_factor = float(raw_weights.sum())
scales = np.array([0.50, 0.75, 0.50])

def logp_components(x):
    x = np.asarray(x)[...,None]
    return np.log(weights) - np.log(scales) - 0.5*np.log(2*np.pi) - 0.5*((x-modes)/scales)**2

def logp(x):
    return logsumexp(logp_components(x), axis=-1)

def grad_logp(x):
    x = np.asarray(x)
    comps = logp_components(x)
    a = np.exp(comps - logsumexp(comps, axis=-1)[...,None])
    comp_grad = -(x[...,None]-modes)/(scales**2)
    return np.sum(a*comp_grad, axis=-1)

target_grid = np.linspace(-12,12,CFG["grid_N"])
p_grid, dxw_grid, _ = normalize_pdf_grid(target_grid, logp(target_grid))
cdf_grid = cdf_from_pdf(target_grid, p_grid)
assert np.all(raw_weights > 0)
assert np.all(weights > 0)
assert np.isclose(weights.sum(), 1.0, atol=1e-14)
assert np.isclose(np.sum(p_grid * dxw_grid), 1.0, atol=2e-6)
assert np.all(np.diff(cdf_grid) >= -1e-12)
assert np.isclose(cdf_grid[-1], 1.0, atol=1e-12)

def grad_logp_with_weights(x, wts):
    x = np.asarray(x)
    comps = (np.log(wts)[None, :] - np.log(scales)[None, :] - 0.5*np.log(2*np.pi)
             - 0.5*((x[..., None] - modes[None, :])/scales[None, :])**2)
    a = np.exp(comps - logsumexp(comps, axis=-1)[..., None])
    comp_grad = -(x[..., None] - modes[None, :])/(scales[None, :]**2)
    return np.sum(a * comp_grad, axis=-1)

raw_logp_grid = logsumexp(
    np.log(raw_weights)[None, :] - np.log(scales)[None, :] - 0.5*np.log(2*np.pi)
    - 0.5*((target_grid[:, None]-modes[None, :])/scales[None, :])**2,
    axis=-1,
)
assert np.allclose(logp(target_grid), raw_logp_grid - np.log(weight_normalization_factor), atol=1e-12, rtol=1e-12)
assert np.allclose(grad_logp(target_grid), grad_logp_with_weights(target_grid, raw_weights), atol=1e-12, rtol=1e-12)
target_mode_mass_precheck = np.array([
    np.sum((np.argmin(np.abs(target_grid[:, None]-modes[None, :]), axis=1) == i) * p_grid * dxw_grid)
    for i in range(3)
])
assert np.isclose(target_mode_mass_precheck.sum(), 1.0, atol=2e-6)
pd.DataFrame([dict(experiment="triple-well", raw_weight_left=float(raw_weights[0]),
                   raw_weight_middle=float(raw_weights[1]), raw_weight_right=float(raw_weights[2]),
                   raw_weight_sum=float(weight_normalization_factor),
                   weight_left=float(weights[0]), weight_middle=float(weights[1]),
                   weight_right=float(weights[2]), weight_sum=float(weights.sum()),
                   target_integral=float(np.sum(p_grid * dxw_grid)),
                   cdf_endpoint=float(cdf_grid[-1]),
                   mode_mass_sum=float(target_mode_mass_precheck.sum()),
                   common_rescaling_changes_drift_and_score=False)]).to_csv(
    TABDIR / "threewell_target_weight_checks.csv", index=False
)
target_rng = np.random.default_rng(GLOBAL_SEED+77)
target_samples = target_grid[weighted_choice_indices(target_rng, p_grid*dxw_grid, 200000 if PROFILE!="smoke" else 20000)]

jump_adjacent = ShellJump1D(a=3.0, h=0.18, lam=1.2, quad_r=CFG["quad_r"], name="adjacent a=3")
jump_overlong = ShellJump1D(a=6.0, h=0.18, lam=1.2, quad_r=CFG["quad_r"], name="overlong a=6")
jumps = {"LSC-adjacent": jump_adjacent, "LSC-overlong": jump_overlong}
validation_grid = np.linspace(-8.0, 8.0, 260)
def raw_logp_fn(x):
    x = np.asarray(x)[..., None]
    return logsumexp(np.log(raw_weights)[None, :] - np.log(scales)[None, :] - 0.5*np.log(2*np.pi)
                     - 0.5*((x-modes[None, :])/scales[None, :])**2, axis=-1)
for _jump in [jump_adjacent, jump_overlong]:
    _score_norm, _ = levy_score_1d_grid(validation_grid, logp, _jump, theta_n=max(4, min(CFG["theta_n"], 10)), log_clip=None, score_clip=None)
    _score_raw, _ = levy_score_1d_grid(validation_grid, raw_logp_fn, _jump, theta_n=max(4, min(CFG["theta_n"], 10)), log_clip=None, score_clip=None)
    assert np.allclose(_score_norm, _score_raw, atol=1e-10, rtol=1e-10)
print(jump_adjacent, jump_overlong)

## 1. Target, CDF, and the two competing jump measures

In [ ]:
rgrid=np.linspace(-7,7,1200)
fig, ax=plt.subplots(1,3,figsize=(14,3.7))
ax[0].plot(target_grid,p_grid,lw=2)
for m in modes: ax[0].axvline(m,color="k",ls="--",lw=0.8,alpha=0.5)
ax[0].set_title("three-well target PDF")
ax[0].set_xlabel("$x$")
ax[1].plot(target_grid,cdf_grid,lw=2)
ax[1].set_title("target CDF")
ax[1].set_xlabel("$x$")
ax[2].plot(rgrid,jump_adjacent.density_on_grid(rgrid),lw=2,label="adjacent $a=3$")
ax[2].plot(rgrid,jump_overlong.density_on_grid(rgrid),lw=2,label="overlong $a=6$")
ax[2].axvspan(3-0.36,3+0.36,color="tab:green",alpha=0.15,label="adjacent displacement")
ax[2].axvspan(6-0.36,6+0.36,color="tab:red",alpha=0.12,label="left-right displacement")
ax[2].set_title(r"jump density $\nu(dr)/\lambda$")
ax[2].set_xlabel("$r$"); ax[2].legend(fontsize=8)
savefig(fig, FIGDIR, "fig01_target_cdf_nu")
plt.show()

## 2. Full gap convergence: adjacent vs overlong

In [ ]:
gap_conv = run_gap_convergence(
    logp_fn=logp, eps=eps, interval=CFG["gap_interval"], Ns=CFG["gap_Ns"],
    jump_map={"Langevin": None, "LSC-adjacent": jump_adjacent, "LSC-overlong": jump_overlong}
)
gap_conv.to_csv(TABDIR/"threewell_gap_convergence.csv", index=False)
display(gap_conv[["method","N_input","N_active","gap","residual","retained_mass_before_renorm","boundary_jump_mass_proxy","elapsed_sec"]])
gap_main=choose_main_gaps(gap_conv)
print("Main gaps:", gap_main)


# Last-two-grid relative change table: a lightweight discretization self-consistency check.
def gap_last_two_relative_change(gap_df):
    rows = []
    for method, sub in gap_df.groupby("method"):
        sub = sub.dropna(subset=["gap"]).sort_values("N_input")
        if len(sub) >= 2:
            prev = float(sub.iloc[-2]["gap"])
            finest = float(sub.iloc[-1]["gap"])
            rel = abs(finest - prev) / max(abs(finest), 1e-14)
            rows.append(dict(method=method, N_previous=int(sub.iloc[-2]["N_input"]),
                             N_finest=int(sub.iloc[-1]["N_input"]),
                             gap_previous=prev, gap_finest=finest,
                             gap_relative_change_last_two=float(rel)))
        elif len(sub) == 1:
            rows.append(dict(method=method, N_previous=np.nan, N_finest=int(sub.iloc[-1]["N_input"]),
                             gap_previous=np.nan, gap_finest=float(sub.iloc[-1]["gap"]),
                             gap_relative_change_last_two=np.nan))
    return pd.DataFrame(rows)

gap_self_consistency = gap_last_two_relative_change(gap_conv)
gap_self_consistency.to_csv(TABDIR / "threewell_gap_last_two_relative_change.csv", index=False)
display(gap_self_consistency)



def run_form_gap_domain_sensitivity(label_prefix, logp_fn, eps, domain_list, N, jump_map):
    rows = []
    for domain in domain_list:
        for method, jump in jump_map.items():
            gap, diag, *_ = full_gap_1d_form(logp_fn, eps, domain[0], domain[1], N=int(N), jump=jump)
            row = dict(diag)
            row.update(dict(method=method, x_min=float(domain[0]), x_max=float(domain[1]), N=int(N), gap=float(gap)))
            rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(TABDIR / f"{label_prefix}form_gap_domain_sensitivity.csv", index=False)
    display(df[["method", "x_min", "x_max", "N_input", "N_active", "gap", "residual", "boundary_jump_mass_proxy"]])
    return df

if PROFILE == "smoke":
    gap_domain_list = [CFG["gap_interval"]]
else:
    gap_domain_list = [(-10.0, 10.0), (-12.0, 12.0), (-15.0, 15.0)]

gap_domain_sensitivity = run_form_gap_domain_sensitivity(
    "threewell_", logp, eps, gap_domain_list, max(CFG["gap_Ns"]),
    {"Langevin": None, "LSC-adjacent": jump_adjacent, "LSC-overlong": jump_overlong}
)



def plot_form_gap_domain_sensitivity(df, label_prefix):
    """Appendix-style visualization of the form-gap domain sensitivity table."""
    if df is None or len(df) == 0:
        return
    d = df.copy()
    d["domain_width"] = d["x_max"] - d["x_min"]
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for method, sub in d.groupby("method"):
        sub = sub.sort_values("domain_width")
        ax[0].plot(sub["domain_width"], sub["gap"], marker=method_marker(method), color=method_color(method), label=method)
        if "boundary_jump_mass_proxy" in sub.columns:
            ax[1].semilogy(sub["domain_width"], np.maximum(sub["boundary_jump_mass_proxy"], 1e-16), marker=method_marker(method), color=method_color(method), label=method)
    ax[0].set_title("form-gap domain sensitivity")
    ax[0].set_xlabel("domain width")
    ax[0].set_ylabel("form gap")
    ax[1].set_title("boundary jump-mass proxy")
    ax[1].set_xlabel("domain width")
    ax[1].set_ylabel("proxy mass")
    for a in ax:
        a.grid(True, alpha=0.25)
        a.legend(fontsize=8)
    savefig(fig, FIGDIR, f"{label_prefix}fig02c_form_gap_domain_sensitivity")
    plt.show()

plot_form_gap_domain_sensitivity(gap_domain_sensitivity, "threewell_")


fig, ax=plt.subplots(1,2,figsize=(11,4))
for method, sub in gap_conv.groupby("method"):
    sub=sub.sort_values("N_input")
    ax[0].plot(sub.N_input, sub.gap, marker="o", label=method)
    ax[1].semilogy(sub.N_input, np.maximum(sub.residual,1e-16), marker="o", label=method)
ax[0].set_title("gap convergence")
ax[0].set_xlabel("N"); ax[0].set_ylabel(r"$\lambda_2$")
ax[1].set_title("eigen residual")
ax[1].set_xlabel("N"); ax[1].set_ylabel("relative residual")
for a in ax: a.legend(fontsize=8)
savefig(fig, FIGDIR, "fig02_gap_convergence")
plt.show()


## 2b. Full generator abscissa and stationarity diagnostics

This cell computes a finite-dimensional diagnostic generator, its spectral abscissa, and stationarity checks needed to interpret that abscissa.  The form gap remains the theorem-facing object.  The abscissa is a diagnostic for this finite-dimensional discretized generator and must be read together with stationary residual, stationary bias, negative-mass diagnostics, and domain/grid sensitivity.


In [ ]:
from scipy.linalg import eig

def gaussian_mass_on_grid(x, dxw, center, scale):
    z = np.exp(-0.5*((x-center)/scale)**2)
    z = z / np.sum(z*dxw)
    return z*dxw

def build_fd_generator_1d(logp_fn, grad_logp_fn, eps, x_min, x_max, N, jump=None,
                          theta_n=12, drift_abs_cap=300.0):
    """Diagnostic backward generator on a finite interval.

    This finite-dimensional generator is used only for abscissa and stationarity diagnostics.
    It matches the notebook convention that outside-domain jump evaluations are clamped to the
    nearest boundary.  Therefore the reported abscissa should be interpreted together with the
    stationarity residual and stationary bias.
    """
    x = np.linspace(float(x_min), float(x_max), int(N))
    p_density, dxw, _ = normalize_pdf_grid(x, logp_fn(x))
    mu = p_density * dxw
    n = len(x)
    dx = x[1] - x[0]
    A = np.zeros((n, n), dtype=float)

    # Diffusion as reflecting nearest-neighbor CTMC.
    rate_diff = eps/(dx*dx)
    for i in range(n):
        if i == 0:
            A[i, i+1] += 2*rate_diff; A[i, i] -= 2*rate_diff
        elif i == n-1:
            A[i, i-1] += 2*rate_diff; A[i, i] -= 2*rate_diff
        else:
            A[i, i-1] += rate_diff; A[i, i+1] += rate_diff; A[i, i] -= 2*rate_diff

    # Upwind drift.  The LSC score is recomputed for the given jump law.
    b = eps * grad_logp_fn(x)
    score_diag = {}
    if jump is not None:
        Sscore, score_diag = levy_score_1d_grid(
            x, logp_fn, jump, theta_n=theta_n, log_clip=None, score_clip=None
        )
        b = b + Sscore
    b = np.clip(b, -drift_abs_cap, drift_abs_cap)
    for i, bi in enumerate(b):
        if bi > 0 and i < n-1:
            r = bi/dx; A[i, i+1] += r; A[i, i] -= r
        elif bi < 0 and i > 0:
            r = -bi/dx; A[i, i-1] += r; A[i, i] -= r

    # Jump operator: lambda E[f(x+r)-f(x)] with clamped interpolation at the finite boundary.
    if jump is not None:
        rs, wr = jump.atoms()
        I = sparse.eye(n, format="csr")
        for r, pr in zip(rs, wr):
            T = interp_shift_matrix_1d(x, float(r))
            A += jump.lam * float(pr) * (T - I).toarray()

    return A, x, mu, dxw, score_diag

def stationary_bias_from_generator(A, mu, neg_tol=1e-8):
    vals, vecs = eig(A.T)
    idx = int(np.argmin(np.abs(vals)))
    v = np.real(vecs[:, idx])
    if np.sum(v) < 0:
        v = -v
    min_entry = float(np.min(v))
    negative_mass = float(np.sum(np.abs(v[v < 0])))
    warning = bool(min_entry < -neg_tol)
    # Do not silently take abs(v): negative mass is a diagnostic of a suspect
    # discrete generator/eigenvector.  We keep a projected debug bias, but the
    # main stationary_bias_l1 is set to NaN whenever the warning triggers.
    v_projected = np.maximum(v, 0.0)
    if np.sum(v_projected) <= 0:
        v_projected = np.ones_like(mu)
        warning = True
    ptilde = v_projected / np.sum(v_projected)
    projected_bias = float(np.sum(np.abs(ptilde - mu)))
    main_bias = np.nan if warning else projected_bias
    diag = {
        "stationary_vector_min": min_entry,
        "stationary_vector_negative_mass": negative_mass,
        "stationary_vector_warning": warning,
        "stationary_bias_l1_projected": projected_bias,
    }
    return ptilde, main_bias, vals[idx], diag


def generator_abscissa_from_A(A, tol=1e-8):
    vals = eig(A, left=False, right=False)
    rates = sorted([-np.real(v) for v in vals if abs(v) > tol and -np.real(v) > tol])
    return float(rates[0]) if rates else np.nan, vals

def run_generator_diagnostics_1d(method_jump_map, domains, Ns, *, label_prefix, init_center, init_scale):
    rows = []
    spectra = {}
    for domain in domains:
        for N in Ns:
            for method, jump in method_jump_map.items():
                A, xg, mug, dxwg, score_diag = build_fd_generator_1d(
                    logp, grad_logp, eps, domain[0], domain[1], int(N), jump=jump,
                    theta_n=max(4, min(CFG.get("theta_n", 12), 14))
                )
                abscissa, vals = generator_abscissa_from_A(A)
                ptilde, bias_l1, zero_eval, stat_diag = stationary_bias_from_generator(A, mug)
                rows.append(dict(
                    method=method, x_min=domain[0], x_max=domain[1], N=int(N),
                    generator_abscissa=abscissa,
                    row_sum_inf=float(np.max(np.abs(A.sum(axis=1)))),
                    stationary_residual_l1=float(np.sum(np.abs(mug @ A))),
                    stationary_residual_linf=float(np.max(np.abs(mug @ A))),
                    stationary_bias_l1=bias_l1,
                    stationary_bias_l1_projected=float(stat_diag.get("stationary_bias_l1_projected", np.nan)),
                    zero_eval_abs=float(abs(zero_eval)),
                    stationary_vector_min=float(stat_diag["stationary_vector_min"]),
                    stationary_vector_negative_mass=float(stat_diag["stationary_vector_negative_mass"]),
                    stationary_vector_warning=bool(stat_diag["stationary_vector_warning"]),
                    score_clip_fraction=float(score_diag.get("score_clip_fraction", 0.0)),
                    logratio_clip_fraction=float(score_diag.get("logratio_clip_fraction", 0.0)),
                ))
                spectra[(method, domain, int(N))] = vals
    gen_df = pd.DataFrame(rows)
    gen_df.to_csv(TABDIR / f"{label_prefix}generator_stationarity_sensitivity.csv", index=False)

    # Main values: largest N on the widest domain in the sweep.
    main_domain = domains[-1]
    main_N = max(Ns)
    main_df = gen_df[(gen_df.N == main_N) & (gen_df.x_min == main_domain[0]) & (gen_df.x_max == main_domain[1])].copy()
    main_df.to_csv(TABDIR / f"{label_prefix}generator_main_diagnostics.csv", index=False)
    display(main_df)

    abscissa = main_df.set_index("method")["generator_abscissa"].to_dict()

    fig, ax = plt.subplots(1, 3, figsize=(14.5, 4.0))
    labels = list(method_jump_map.keys())
    xloc = np.arange(len(labels))
    form_vals = [gap_main.get(m, np.nan) for m in labels]
    abs_vals = [abscissa.get(m, np.nan) for m in labels]
    w = 0.36
    ax[0].bar(xloc-w/2, form_vals, width=w, color=[method_color(m) for m in labels], alpha=0.9, hatch="//", edgecolor="white", label="form gap")
    ax[0].bar(xloc+w/2, abs_vals, width=w, color=[method_color(m) for m in labels], alpha=0.9, edgecolor="white", label="generator abscissa")
    ax[0].set_xticks(xloc); ax[0].set_xticklabels(labels, rotation=20)
    ax[0].set_title("form gap and abscissa")
    ax[0].set_ylabel("rate")
    ax[0].legend(fontsize=8)

    ax[1].bar(main_df.method, main_df.stationary_bias_l1, color=[method_color(m) for m in main_df.method])
    ax[1].set_title("discrete stationary bias")
    ax[1].set_ylabel("L1 distance")
    ax[1].tick_params(axis="x", rotation=20)

    for method in labels:
        sub = gen_df[gen_df.method == method].sort_values(["x_max", "N"])
        ax[2].plot(sub.N.astype(str) + " @ [" + sub.x_min.astype(str) + "," + sub.x_max.astype(str) + "]",
                   sub.generator_abscissa, marker=method_marker(method), color=method_color(method), label=method)
    ax[2].set_title("N and domain sensitivity")
    ax[2].set_ylabel("abscissa")
    ax[2].tick_params(axis="x", rotation=45, labelsize=7)
    ax[2].legend(fontsize=8)
    savefig(fig, FIGDIR, f"{label_prefix}fig02b_abscissa_stationarity_sensitivity")
    plt.show()

    return gen_df, main_df, abscissa



# Main diagnostic generator: Langevin and the two LSC jump laws.
if PROFILE == "smoke":
    GEN_NS = [45]
    GEN_DOMAINS = [(-10.0, 10.0)]
elif PROFILE == "paperlite":
    GEN_NS = [90, 140]
    GEN_DOMAINS = [(-10.0, 10.0), (-15.0, 15.0)]
else:
    GEN_NS = [120, 180, 260]
    GEN_DOMAINS = [(-10.0, 10.0), (-12.0, 12.0), (-15.0, 15.0)]

generator_diag_df, generator_main_df, abscissa_main = run_generator_diagnostics_1d(
    {"Langevin": None, "LSC-adjacent": jump_adjacent, "LSC-overlong": jump_overlong},
    GEN_DOMAINS, GEN_NS, label_prefix="threewell_", init_center=-3.0, init_scale=0.08
)
print("Main generator abscissae:", abscissa_main)

In [ ]:

# Extra three-well spectral hierarchy diagnostic: two low modes should separate from the faster intra-well modes.
required_cols = ["lambda2", "lambda3", "lambda4", "lambda5", "ratio32", "ratio43"]
missing = [c for c in required_cols if c not in gap_conv.columns]
if missing:
    print("Missing columns for spectrum hierarchy diagnostic:", missing)
else:
    hierarchy_cols = ["method", "N_input", "lambda2", "lambda3", "lambda4", "lambda5", "ratio32", "ratio43", "ratio54"]
    hierarchy_df = gap_conv[hierarchy_cols].copy()
    hierarchy_df.to_csv(TABDIR / "threewell_gap_spectral_hierarchy.csv", index=False)
    display(hierarchy_df)

    fig, ax = plt.subplots(1, 4, figsize=(16.0, 3.9))
    for method, sub in gap_conv.groupby("method"):
        sub = sub.sort_values("N_input")
        color = method_color(method)
        marker = method_marker(method)
        ax[0].plot(sub.N_input, sub.lambda2, marker=marker, color=color, label=method)
        ax[1].plot(sub.N_input, sub.lambda3, marker=marker, color=color, label=method)
        ax[2].plot(sub.N_input, sub.lambda4, marker=marker, color=color, label=method)
        ax[3].plot(sub.N_input, sub.ratio32, marker=marker, color=color, label=method)
    ax[0].set_title("lambda2: slowest nonzero mode")
    ax[1].set_title("lambda3: second slow mode")
    ax[2].set_title("lambda4: first faster mode")
    ax[3].set_title("lambda3 / lambda2")
    for a in ax:
        a.set_xlabel("grid size N")
        a.grid(alpha=0.25)
    ax[0].set_ylabel("form eigenvalue")
    ax[3].set_ylabel("ratio")
    ax[0].legend(fontsize=7)
    savefig(fig, FIGDIR, "fig02_spectrum_two_slow_modes")
    plt.show()


## Diagnostic: closed-form Gaussian-mixture Lévy score

The 3-well target is a one-dimensional Gaussian mixture.  The Lévy-score integral over the interpolation parameter can therefore be evaluated in closed form for each Gaussian component and each jump quadrature atom.  The main sampler continues to use the original quadrature score from v14; this cell only compares the closed-form score against the quadrature implementation.


In [ ]:
def _theta_integral_exp_quadratic(a, b):
    # Integral from 0 to 1 of exp(a theta - b theta^2) dtheta.
    # Uses erfcx branches to avoid cancellation for large positive or negative arguments.
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    out = np.empty_like(a, dtype=float)
    small = b < 1e-14
    if np.any(small):
        aa = a[small]
        out[small] = np.where(np.abs(aa) < 1e-10, 1.0, np.expm1(np.clip(aa, -700, 700))/aa)
    if np.any(~small):
        aa = a[~small]
        bb = b[~small]
        sb = np.sqrt(bb)
        z0 = -aa/(2*sb)
        z1 = sb - aa/(2*sb)
        pref = np.sqrt(np.pi)/(2*sb)
        val = np.empty_like(aa)
        pos = (z0 > 5.0) & (z1 > 5.0)
        neg = (z0 < -5.0) & (z1 < -5.0)
        mid = ~(pos | neg)
        if np.any(pos):
            val[pos] = pref[pos] * (erfcx(z0[pos]) - np.exp(np.clip(aa[pos]-bb[pos], -700, 700))*erfcx(z1[pos]))
        if np.any(neg):
            val[neg] = pref[neg] * (np.exp(np.clip(aa[neg]-bb[neg], -700, 700))*erfcx(-z1[neg]) - erfcx(-z0[neg]))
        if np.any(mid):
            val[mid] = pref[mid] * np.exp(np.clip(aa[mid]*aa[mid]/(4*bb[mid]), -700, 700)) * (erf(z1[mid]) - erf(z0[mid]))
        out[~small] = np.maximum(val, 0.0)
    return out
def levy_score_1d_mog_closed_form(xgrid, jump: ShellJump1D, log_clip=None, score_clip=None):
    x = np.asarray(xgrid, dtype=float)
    lp_x = logp(x)
    rs, wr = jump.atoms()
    S = np.zeros_like(x)
    nonfinite = 0
    score_changing_clip = 0
    overflow_guard_clip = 0
    total_terms = 0
    float_log_clip = float(np.log(np.finfo(float).max) - 2.0)
    effective_log_clip = float_log_clip if log_clip is None else min(float(log_clip), float_log_clip)
    for r, pr in zip(rs, wr):
        d = x[:, None] - modes[None, :]
        sc2 = scales[None, :]**2
        a = d * r / sc2
        b = np.full_like(a, (r*r)/(2.0)) / sc2
        log_pref = np.log(weights)[None, :] - np.log(scales)[None, :] - 0.5*np.log(2*np.pi) - 0.5*(d*d)/sc2 - lp_x[:, None]
        integ = _theta_integral_exp_quadratic(a, b)
        # Correct closed form: combine the Gaussian-component prefactor and the
        # theta integral in log space.  Clipping log_pref before multiplying by
        # integ is wrong in the tails because log_pref can be very negative while
        # integ is exponentially large; their product is the stable quantity.
        log_terms = log_pref + np.log(np.maximum(integ, 1e-300))
        nonfinite += int(np.sum(~np.isfinite(log_terms)))
        overflow_guard_clip += int(np.sum(np.abs(log_terms) > float_log_clip))
        if log_clip is not None:
            score_changing_clip += int(np.sum(np.abs(log_terms) > effective_log_clip))
        total_terms += log_terms.size
        log_terms = np.nan_to_num(log_terms, nan=-np.inf, posinf=effective_log_clip, neginf=-np.inf)
        ratio_int = np.exp(np.clip(logsumexp(log_terms, axis=1), -effective_log_clip, effective_log_clip))
        S += pr * r * ratio_int
    S = -jump.lam * S
    raw = S.copy()
    if score_clip is None:
        clip_fraction = 0.0
    else:
        S = np.clip(S, -score_clip, score_clip)
        clip_fraction = float(np.mean(np.abs(raw) > score_clip))
    return S, {"score_clip_fraction": clip_fraction,
               "score_changing_logratio_clip_fraction": float(score_changing_clip / max(1, total_terms)),
               "overflow_guard_logratio_clip_fraction": float(overflow_guard_clip / max(1, total_terms)),
               "effective_log_clip": float(effective_log_clip),
               "closed_form_nonfinite_fraction": float(nonfinite / max(1, total_terms)),
               "score_backend": "gaussian_mixture_closed_form_theta"}

def compare_closed_form_score_1d(jump, name):
    ncmp = 240 if PROFILE == "smoke" else 700
    xcmp = np.linspace(x_clip[0], x_clip[1], ncmp)
    theta_ref = 40 if PROFILE == "smoke" else 96
    q, qdiag = levy_score_1d_grid(xcmp, logp, jump, theta_n=theta_ref, log_clip=None, score_clip=None)
    cf, cfdiag = levy_score_1d_mog_closed_form(xcmp, jump, log_clip=None, score_clip=None)
    err = np.abs(cf - q)
    # Target-weighted error avoids letting far-tail points with negligible mass
    # dominate the diagnostic.  The max error is still reported separately.
    p_cmp, dxw_cmp, _ = normalize_pdf_grid(xcmp, logp(xcmp))
    w = p_cmp * dxw_cmp
    diag = dict(method=name, theta_ref=theta_ref, max_abs_error=float(np.max(err)),
                mean_abs_error=float(np.mean(err)), target_weighted_abs_error=float(np.sum(w*err)),
                relative_l2_error=float(np.linalg.norm(cf-q)/max(np.linalg.norm(q),1e-300)),
                target_weighted_relative_l2=float(np.sqrt(np.sum(w*(cf-q)**2))/max(np.sqrt(np.sum(w*q*q)),1e-300)),
                quad_score_clip=qdiag.get("score_clip_fraction", np.nan),
                cf_score_clip=cfdiag.get("score_clip_fraction", np.nan))
    return xcmp, q, cf, err, diag

score_compare_rows=[]
fig, ax = plt.subplots(2, 2, figsize=FIGSIZE_GRID, sharex=True)
for row, (name, jump) in enumerate([("LSC-adjacent", jump_adjacent), ("LSC-overlong", jump_overlong)]):
    xcmp, q, cf, err, diag = compare_closed_form_score_1d(jump, name)
    score_compare_rows.append(diag)
    ax[row,0].plot(xcmp, q, color="0.25", lw=1.6, label="theta quadrature")
    ax[row,0].plot(xcmp, cf, color=method_color(name), lw=1.2, ls="--", label="closed form")
    ax[row,0].set_title(f"{name}: Lévy score")
    ax[row,0].set_ylabel("score")
    ax[row,0].legend(fontsize=7)
    ax[row,1].semilogy(xcmp, np.maximum(err, 1e-14), color=method_color(name), lw=1.3)
    ax[row,1].set_title(f"{name}: absolute difference")
    ax[row,1].set_ylabel("abs error")
for a in ax.ravel():
    a.set_xlabel("x")
score_compare_df = pd.DataFrame(score_compare_rows)
score_compare_df.to_csv(TABDIR / "threewell_closed_form_score_comparison.csv", index=False)
savefig(fig, FIGDIR, "fig01b_closed_form_levy_score_comparison")
plt.show()
display(score_compare_df)


## 3. Samplers and common initial cloud

The initial cloud is intentionally core-localized.  This probes inter-well exploration from a metastable initial condition.  Early-time local relaxation of the cloud should not be interpreted as inter-well mixing.



In [ ]:
def kinetic_langevin_step_1d(x, p, dt, rng, gamma=0.65, mass=0.02):
    """BAOAB kinetic Langevin step for 1D targets with mass-scaled velocity."""
    force = eps * grad_logp(x)
    p = p + 0.5 * dt * force
    x = x + 0.5 * dt * (p / mass)
    a = math.exp(-gamma * dt)
    p = a * p + math.sqrt(eps * mass * (1.0 - a*a)) * rng.standard_normal(p.shape)
    x = x + 0.5 * dt * (p / mass)
    force = eps * grad_logp(x)
    p = p + 0.5 * dt * force
    return x, p

def classify_modes_1d(x):
    x=np.asarray(x)
    return np.argmin(np.abs(x[:,None]-modes[None,:]),axis=1)

target_mode_mass=np.array([np.sum((classify_modes_1d(target_grid)==i)*p_grid*dxw_grid) for i in range(3)])
print("target mode mass", target_mode_mass)

# Cache Levy scores once per LSC jump law.
score_cache = {}
for _method, _jump in [("LSC-adjacent", jump_adjacent), ("LSC-overlong", jump_overlong)]:
    sg=np.linspace(x_clip[0],x_clip[1],1400 if PROFILE!="smoke" else 500)
    sv, sd=levy_score_1d_grid(sg, logp, _jump, theta_n=CFG["theta_n"], log_clip=None, score_clip=None)
    score_cache[_method]=(sg,sv,sd)
print("cached 1D LSC quadrature score diagnostics:", {k:v[2] for k,v in score_cache.items()})

def simulate_1d_three(method, rng_seed, x0, seed_id=None, *, score_cache=None,
                      dt_override=None, n_steps_override=None, rec_override=None):
    rng=np.random.default_rng(rng_seed)
    dt=CFG["dt"] if dt_override is None else float(dt_override)
    n_steps=CFG["n_steps"] if n_steps_override is None else int(n_steps_override)
    rec=CFG["record_every"] if rec_override is None else int(rec_override)
    x=x0.copy(); kin_mass = CFG.get("kinetic_mass", 0.02)
    p = math.sqrt(eps * kin_mass) * rng.standard_normal(x.shape) if method == "Kinetic-Langevin" else None; times=[]; samples=[]; accs=[]; clip_sum=0.0; clip_max=0.0; clip_count=0; score_diag={}
    jump = None
    if method=="LSC-adjacent": jump=jump_adjacent
    if method=="LSC-overlong": jump=jump_overlong
    score_grid=None
    if jump is not None:
        if score_cache is not None and method in score_cache:
            score_grid=score_cache[method]
            score_diag=score_grid[2]
        else:
            sg=np.linspace(x_clip[0],x_clip[1],1400 if PROFILE!="smoke" else 500)
            sv, score_diag=levy_score_1d_mog_closed_form(sg, jump, log_clip=None, score_clip=None)
            score_grid=(sg,sv,score_diag)
    for k in range(n_steps+1):
        if k%rec==0:
            times.append(k*dt); samples.append(x.copy())
        if k==n_steps: break
        if method=="Langevin":
            x=x+dt*eps*grad_logp(x)+np.sqrt(2*eps*dt)*rng.standard_normal(x.shape)
        elif method=="Kinetic-Langevin":
            x,p=kinetic_langevin_step_1d(x,p,dt,rng,gamma=CFG.get("kinetic_gamma",0.65), mass=CFG.get("kinetic_mass",0.02))
        elif method in ["LSC-adjacent","LSC-overlong"]:
            sg,sv,_=score_grid
            S=np.interp(np.clip(x,sg[0],sg[-1]),sg,sv)
            drift=eps*grad_logp(x)+S
            x=x+dt*drift/(1+dt*np.abs(drift))+np.sqrt(2*eps*dt)*rng.standard_normal(x.shape)
            x, nj = apply_compound_poisson_jumps_1d(x, jump, rng, dt)
        elif method=="CP-overlong":
            jump=jump_overlong
            x=x+dt*eps*grad_logp(x)+np.sqrt(2*eps*dt)*rng.standard_normal(x.shape)
            x, nj = apply_compound_poisson_jumps_1d(x, jump, rng, dt)
        x, _clip_fraction = safety_clip_1d(x, method, x_clip)
        clip_sum += float(_clip_fraction); clip_max = max(clip_max, float(_clip_fraction)); clip_count += 1
    return {"method":method,"seed": seed_id if seed_id is not None else rng_seed, "rng_seed": rng_seed, "times":np.array(times),"samples":samples,
            "accept":np.nan, "score_diag":score_diag,
            "safety_clip_fraction": float(clip_sum/max(1, clip_count)),
            "safety_clip_fraction_max": float(clip_max)}

methods=["Langevin","Kinetic-Langevin","LSC-adjacent","LSC-overlong","CP-overlong"]
base_rng=np.random.default_rng(GLOBAL_SEED+200)
tasks=[]
for s in range(CFG["n_seeds"]):
    # A tight left-well initial cloud makes the initial condition cleaner and shortens transient ambiguity.
    x0=make_initial_cloud_1d(base_rng, center=-3.0, scale=0.15, n=CFG["n_particles"], clip=x_clip)
    for mi,m in enumerate(methods):
        tasks.append((m, GLOBAL_SEED+3000*s+mi, x0.copy(), s))

if HAS_JOBLIB and CFG.get("n_jobs", 1) > 1:
    all_runs = Parallel(n_jobs=CFG["n_jobs"], backend="loky")(
        delayed(simulate_1d_three)(m, rs, x0, seed_id=sid, score_cache=score_cache)
        for (m, rs, x0, sid) in tasks
    )
else:
    all_runs=[simulate_1d_three(m, rs, x0, seed_id=sid, score_cache=score_cache) for (m, rs, x0, sid) in tasks]
print("runs:",len(all_runs), "n_jobs:", CFG.get("n_jobs", 1))
print("Kinetic-Langevin gamma/mass:", CFG.get("kinetic_gamma", 0.65), CFG.get("kinetic_mass", 0.02))
print("LSC score diag sample:", [r["score_diag"] for r in all_runs if "LSC" in r["method"]][:2])


## 4. Mode mixing, missed-middle visualization, final PDF/CDF

In [ ]:

metric_rows = []
for r in all_runs:
    for t, xs in zip(r["times"], r["samples"]):
        modes_emp = classify_modes_1d(xs)
        mass = np.array([np.mean(modes_emp == i) for i in range(3)])
        mode_tv = 0.5 * np.sum(np.abs(mass - target_mode_mass))
        middle_error = abs(mass[1] - target_mode_mass[1])
        ecdf = empirical_cdf_on_grid(xs, target_grid)
        cdf_sup = np.max(np.abs(ecdf - cdf_grid))
        centers, hp = empirical_pdf_1d(xs, target_grid, clip_range=(target_grid[0], target_grid[-1]))
        l1 = _trapz(np.abs(hp - np.interp(centers, target_grid, p_grid)), centers)
        w1 = w1_empirical_to_target_cdf(xs, target_grid, cdf_grid)
        metric_rows.append(dict(method=r["method"], seed=r["seed"], time=t, mode_TV=mode_tv,
                                middle_error=middle_error, CDF_sup=cdf_sup, L1_pdf=l1, W1=w1,
                                m0=mass[0], m1=mass[1], m2=mass[2]))
metrics = pd.DataFrame(metric_rows)
metrics.to_csv(TABDIR / "threewell_metrics_timeseries.csv", index=False)

# Fairness diagnostic: all methods share the same initial cloud per seed.
init_rows = []
for r in all_runs:
    xs0 = r["samples"][0]
    cl0 = classify_modes_1d(xs0)
    init_rows.append(dict(method=r["method"], seed=r["seed"], mean=float(np.mean(xs0)),
                          std=float(np.std(xs0)), left=float(np.mean(cl0 == 0)),
                          middle=float(np.mean(cl0 == 1)), right=float(np.mean(cl0 == 2))))
init_df = pd.DataFrame(init_rows)
display(init_df)
for col in ["left", "middle", "right"]:
    piv = init_df.pivot(index="seed", columns="method", values=col)
    print(f"max within-seed spread of initial {col} mass:", float((piv.max(axis=1)-piv.min(axis=1)).max()))
for metric_name in ["W1", "mode_TV", "CDF_sup"]:
    piv = metrics[metrics.time == metrics.time.min()].pivot_table(index="seed", columns="method", values=metric_name)
    print(f"max within-seed spread of initial {metric_name} across methods:", float((piv.max(axis=1)-piv.min(axis=1)).max()))

fig, ax = plt.subplots(2, 2, figsize=(13.2, 8.2), sharex=True)
ax = ax.ravel()
panels = [
    ("mode_TV", "mode-mass TV decay"),
    ("middle_error", "middle-well absolute error"),
    ("W1", "Wasserstein-1"),
    ("CDF_sup", "CDF sup error"),
]
for j, (metric, title) in enumerate(panels):
    for m in methods:
        plot_metric_with_ci(ax[j], metrics, m, metric, logy=True, lw=1.5)
    for m in ["Langevin", "LSC-adjacent", "LSC-overlong"]:
        add_rate_reference_lines(
            ax[j], metrics, metric, m,
            form_rate=gap_main.get(m, np.nan),
            abscissa_rate=globals().get("abscissa_main", {}).get(m, np.nan),
            logy=True,
        )
    ax[j].set_title(title)
    ax[j].set_xlabel("time")
    ax[j].set_xlim(metrics.time.min(), metrics.time.max())
    ax[j].set_ylabel("error")
    ax[j].legend(fontsize=5.8, ncol=2)
savefig(fig, FIGDIR, "fig03_mode_mixing_with_form_and_abscissa_references")
plt.show()

final_by = {}
pdf_curves = {}
cdf_curves = {}
for m in methods:
    final = np.concatenate([r["samples"][-1] for r in all_runs if r["method"] == m])
    final_by[m] = final
    centers, hp = empirical_pdf_1d(final, target_grid, clip_range=(target_grid[0], target_grid[-1]))
    pdf_curves[m] = (centers, hp)
    cdf_curves[m] = empirical_cdf_on_grid(final, target_grid)

pdf_ymax = 1.08 * max([np.max(p_grid)] + [np.max(v[1]) for v in pdf_curves.values()])
xlim_pdf = (target_grid[0], target_grid[-1])

fig, ax = plt.subplots(2, 3, figsize=(15.2, 8.6))
ax = ax.ravel()
ax[0].plot(target_grid, p_grid, color="k", lw=2.8, label="target PDF")
ax[0].fill_between(target_grid, 0, p_grid, color="0.4", alpha=0.12)
for mm in modes:
    ax[0].axvline(mm, color="k", ls="--", lw=0.7, alpha=0.4)
ax[0].set_title("true target PDF")
ax[0].set_xlim(xlim_pdf); ax[0].set_ylim(0, pdf_ymax)
ax[0].legend(fontsize=8)

for m in methods:
    centers, hp = pdf_curves[m]
    ax[1].plot(centers, hp, lw=1.5, color=method_color(m), label=m)
ax[1].plot(target_grid, p_grid, color="k", lw=2.5, label="target")
ax[1].set_title("final empirical PDF vs target")
ax[1].set_xlim(xlim_pdf); ax[1].set_ylim(0, pdf_ymax)
ax[1].legend(fontsize=7)

ax[2].plot(target_grid, cdf_grid, color="k", lw=2.8, label="target CDF")
for m in methods:
    ax[2].plot(target_grid, cdf_curves[m], lw=1.5, color=method_color(m), label=m)
ax[2].set_title("final empirical CDF vs target")
ax[2].set_xlim(xlim_pdf); ax[2].set_ylim(-0.02, 1.02)
ax[2].legend(fontsize=7)

mass_df = []
for m, final in final_by.items():
    cl = classify_modes_1d(final)
    mass_df.append([m] + [np.mean(cl == i) for i in range(3)])
mass_df = pd.DataFrame(mass_df, columns=["method", "left", "middle", "right"]).set_index("method")
mass_df.plot(kind="bar", ax=ax[3])
for i, val in enumerate(target_mode_mass):
    ax[3].axhline(val, color="k", ls=":", lw=0.7, alpha=0.5)
ax[3].set_title("terminal mode masses")
ax[3].tick_params(axis="x", rotation=25)

t_final = metrics.time.max()
term_stats = (metrics[np.isclose(metrics.time, t_final)]
              .groupby("method")[["W1", "CDF_sup", "mode_TV", "middle_error", "L1_pdf"]]
              .agg(["mean", "std"]))
term_mean = term_stats.xs("mean", level=1, axis=1)
term_mean.plot(kind="bar", ax=ax[4])
ax[4].set_title("appendix weak diagnostics")
ax[4].tick_params(axis="x", rotation=25)
term = term_stats.copy()
term.columns = [f"{metric}_{stat}" for metric, stat in term.columns]
term = term.reset_index()

snapshot_times = np.linspace(0, metrics.time.max(), 5)
for m in methods:
    run0 = next(r for r in all_runs if r["method"] == m)
    ts = run0["times"]
    chosen = [int(np.argmin(np.abs(ts - tt))) for tt in snapshot_times]
    for ci in chosen:
        xs = run0["samples"][ci]
        sub = xs[::max(1, len(xs)//350)]
        jitter = 0.06 * np.random.default_rng(323 + ci).standard_normal(len(sub))
        ax[5].scatter(sub, np.full_like(sub, ts[ci]) + jitter, s=4, alpha=0.30, color=method_color(m), label=m if ci == chosen[0] else None)
ax[5].set_title("particle cloud snapshots, one representative seed")
ax[5].set_xlabel("x"); ax[5].set_ylabel("time")
ax[5].set_xlim(xlim_pdf); ax[5].set_ylim(metrics.time.min()-0.05, metrics.time.max()+0.05)
ax[5].legend(fontsize=6, ncol=2)

savefig(fig, FIGDIR, "fig04_final_pdf_cdf_modes_particles")
plt.show()
term.to_csv(TABDIR / "threewell_terminal_metrics.csv", index=False)
display(term)


In [ ]:
# ---------------------------------------------------------------------
# Phase17H density-level convergence, KDE/CDF, histogram-bin, and drift diagnostics.
# ---------------------------------------------------------------------
density_bandwidth = target_local_scott_bandwidth(target_grid, p_grid, CFG["n_particles"])
chi_lo, chi_hi, chi_omitted = central_interval_from_cdf(target_grid, cdf_grid, mass=0.98)
chi_mask = (target_grid >= chi_lo) & (target_grid <= chi_hi)
density_config = DensityDiagnosticConfig(
    bandwidth=float(density_bandwidth),
    chi_interval=(float(chi_lo), float(chi_hi)),
    chi_omitted_mass=float(chi_omitted),
    chi_target_min=float(np.min(p_grid[chi_mask])),
)
density_meta = pd.DataFrame([dict(
    experiment="triple-well",
    grid_left=float(target_grid[0]),
    grid_right=float(target_grid[-1]),
    grid_N=int(len(target_grid)),
    target_integral=float(np.sum(p_grid * dxw_grid)),
    target_cdf_endpoint=float(cdf_grid[-1]),
    omitted_tail_mass_note="not estimated; displayed grid is the simulation clipping domain",
    KDE_bandwidth=float(density_bandwidth),
    bandwidth_rule="target-local Scott rule on high-density connected components, floored at two grid spacings",
    chi_interval_left=float(chi_lo),
    chi_interval_right=float(chi_hi),
    chi_omitted_target_mass=float(chi_omitted),
    chi_target_density_min=float(np.min(p_grid[chi_mask])),
    quadrature=density_config.quadrature,
)])
bias_floor_rows = estimate_kde_bias_floor(
    target_grid, p_grid, density_bandwidth, CFG["n_particles"], density_config,
    n_replicates=12 if PROFILE != "smoke" else 3, seed=GLOBAL_SEED + 1708,
)
density_bias_floor_df = pd.DataFrame(bias_floor_rows)
density_bias_floor_summary = density_bias_floor_df.drop(columns=["replicate"]).agg(["mean", "std", "max"]).reset_index().rename(columns={"index": "summary"})
density_bias_floor_df.to_csv(TABDIR / "threewell_kde_bias_floor_by_reference_sample.csv", index=False)
density_bias_floor_summary.to_csv(TABDIR / "threewell_kde_bias_floor_summary.csv", index=False)
density_meta["KDE_bias_floor_chi2_mean"] = float(density_bias_floor_df["truncated_KDE_chi2"].mean())
density_meta["KDE_bias_floor_chi2_max"] = float(density_bias_floor_df["truncated_KDE_chi2"].max())
density_meta.to_csv(TABDIR / "threewell_density_diagnostic_metadata.csv", index=False)

density_rows = []
hist_edges = np.linspace(float(target_grid[0]), float(target_grid[-1]), 75)
kde_hist_rows = []
for r in all_runs:
    for t, xs in zip(r["times"], r["samples"]):
        kde, kde_diag = binned_gaussian_kde_on_grid_with_diagnostics(xs, target_grid, density_bandwidth)
        metric_bundle = density_metric_bundle(kde, p_grid, target_grid, density_config, samples=xs)
        density_rows.append(dict(
            experiment="triple-well",
            method=r["method"],
            seed=r["seed"],
            time=float(t),
            KDE_bandwidth=float(density_bandwidth),
            KDE_integral=float(kde_diag["integral"]),
            KDE_integral_before_grid_renormalization=float(kde_diag["integral_before_grid_renormalization"]),
            KDE_tail_mass_outside_plot=float(kde_diag["tail_mass_outside_grid"]),
            **{k: float(v) for k, v in metric_bundle.items()},
            chi_interval_left=float(chi_lo),
            chi_interval_right=float(chi_hi),
            chi_omitted_target_mass=float(chi_omitted),
            chi_target_density_min=float(np.min(p_grid[chi_mask])),
            quadrature=density_config.quadrature,
        ))
        comp = kde_histogram_bin_mass_l1(xs, target_grid, density_bandwidth, hist_edges)
        kde_hist_rows.append(dict(
            experiment="triple-well",
            method=r["method"],
            seed=r["seed"],
            time=float(t),
            KDE_bandwidth=float(density_bandwidth),
            **comp,
        ))
density_df = pd.DataFrame(density_rows)
assert np.all(np.isfinite(density_df[["KDE_integral", "KDE_tail_mass_outside_plot", "L1_density_error", "L2_density_error", "truncated_KDE_chi2", "KDE_CDF_sup_error"]].to_numpy()))
assert float(density_df["L2_density_error"].min()) >= -1e-14
assert float(density_df["truncated_KDE_chi2"].min()) >= -1e-14
assert density_df.groupby(["method", "seed", "time"])["KDE_bandwidth"].nunique().max() == 1
density_df.to_csv(TABDIR / "threewell_density_convergence_by_seed.csv", index=False)
density_summary_df = summarize_density_diagnostics(density_rows)
density_summary_df.to_csv(TABDIR / "threewell_density_convergence_summary.csv", index=False)

kde_hist_df = pd.DataFrame(kde_hist_rows)
assert np.all(np.isfinite(kde_hist_df[["histogram_integral", "KDE_integral", "KDE_tail_mass_outside_plot", "KDE_vs_histogram_bin_mass_L1"]].to_numpy()))
assert np.allclose(kde_hist_df["histogram_integral"], 1.0, atol=1e-12)
assert np.allclose(kde_hist_df["KDE_integral"], 1.0, atol=5e-3)
kde_hist_df.to_csv(TABDIR / "threewell_kde_histogram_diagnostics.csv", index=False)
display(kde_hist_df.groupby("method")[["KDE_tail_mass_outside_plot", "KDE_vs_histogram_bin_mass_L1"]].mean().reset_index())

rep_seed = sorted(metrics["seed"].unique())[0]
snapshot_times = np.array([metrics.time.min(), 0.5*(metrics.time.min()+metrics.time.max()), metrics.time.max()])
fig, axes = plt.subplots(2, 3, figsize=(15.2, 7.8), sharex=True, sharey=True)
axes = axes.ravel()
for ax, m in zip(axes, methods):
    run0 = next(r for r in all_runs if r["method"] == m and r["seed"] == rep_seed)
    ts = run0["times"]
    ax.plot(target_grid, p_grid, color="black", lw=2.0, label="target")
    for tt, ls in zip(snapshot_times, [":", "--", "-"]):
        idx = int(np.argmin(np.abs(ts - tt)))
        kde, _ = binned_gaussian_kde_on_grid_with_diagnostics(run0["samples"][idx], target_grid, density_bandwidth)
        ax.plot(target_grid, kde, lw=1.2, ls=ls, label=f"t={ts[idx]:.2f}")
    ax.set_title(m)
    ax.set_xlabel("x")
    ax.set_ylabel("density")
    ax.set_xlim(x_clip)
    ax.legend(fontsize=7)
savefig(fig, FIGDIR, "fig06_density_kde_snapshots")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=FIGSIZE_ROW)
for m in methods:
    sub = density_summary_df[density_summary_df.method == m]
    y = np.maximum(sub["L2_density_error_mean"].to_numpy(dtype=float), 1e-14)
    se = sub["L2_density_error_se"].fillna(0.0).to_numpy(dtype=float)
    t = sub["time"].to_numpy(dtype=float)
    ax.semilogy(t, y, marker=method_marker(m), color=method_color(m), lw=1.6, ms=3, label=m)
    ax.fill_between(t, np.maximum(y-2*se, 1e-14), y+2*se, color=method_color(m), alpha=0.10, linewidth=0)

for m in ['Langevin', 'LSC-adjacent', 'LSC-overlong']:
    density_reference_df = density_summary_df[["method", "time", "L2_density_error_mean"]].rename(
        columns={"L2_density_error_mean": "L2_density_error"}
    )
    add_rate_reference_lines(
        ax,
        density_reference_df,
        "L2_density_error",
        m,
        form_rate=gap_main.get(m, np.nan),
        abscissa_rate=globals().get("abscissa_main", {}).get(m, np.nan),
        logy=True,
    )

ax.set_xlabel("time"); ax.set_ylabel("L2 density error")
ax.set_title("KDE density convergence: L2 error")
ax.legend(fontsize=8, ncol=2)
savefig(fig, FIGDIR, "fig06b_l2_density_error")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=FIGSIZE_ROW)
for m in methods:
    sub = density_summary_df[density_summary_df.method == m]
    y = np.maximum(sub["truncated_KDE_chi2_mean"].to_numpy(dtype=float), 1e-14)
    se = sub["truncated_KDE_chi2_se"].fillna(0.0).to_numpy(dtype=float)
    t = sub["time"].to_numpy(dtype=float)
    ax.semilogy(t, y, marker=method_marker(m), color=method_color(m), lw=1.6, ms=3, label=m)
    ax.fill_between(t, np.maximum(y-2*se, 1e-14), y+2*se, color=method_color(m), alpha=0.10, linewidth=0)

for m in ['Langevin', 'LSC-adjacent', 'LSC-overlong']:
    density_reference_df = density_summary_df[["method", "time", "truncated_KDE_chi2_mean"]].rename(
        columns={"truncated_KDE_chi2_mean": "truncated_KDE_chi2"}
    )
    add_rate_reference_lines(
        ax,
        density_reference_df,
        "truncated_KDE_chi2",
        m,
        form_rate=gap_main.get(m, np.nan),
        abscissa_rate=globals().get("abscissa_main", {}).get(m, np.nan),
        logy=True,
    )

ax.set_xlabel("time"); ax.set_ylabel("truncated KDE chi-square")
ax.set_title("KDE density convergence: truncated chi-square diagnostic")
ax.legend(fontsize=8, ncol=2)
savefig(fig, FIGDIR, "fig06c_truncated_kde_chi2")
plt.show()

sens_rows = []
for factor in [0.75, 1.00, 1.25]:
    h = factor * density_bandwidth
    cfg_s = DensityDiagnosticConfig(
        bandwidth=float(h),
        chi_interval=density_config.chi_interval,
        chi_omitted_mass=density_config.chi_omitted_mass,
        chi_target_min=density_config.chi_target_min,
    )
    for r in all_runs:
        xs = r["samples"][-1]
        kde, kde_diag = binned_gaussian_kde_on_grid_with_diagnostics(xs, target_grid, h)
        metric_bundle = density_metric_bundle(kde, p_grid, target_grid, cfg_s, samples=xs)
        sens_rows.append(dict(experiment="triple-well", method=r["method"], seed=r["seed"],
                              time=float(r["times"][-1]), bandwidth_factor=float(factor),
                              KDE_bandwidth=float(h), KDE_integral=float(kde_diag["integral"]),
                              **{k: float(v) for k, v in metric_bundle.items()}))
density_sensitivity_df = pd.DataFrame(sens_rows)
density_sensitivity_df.to_csv(TABDIR / "threewell_density_bandwidth_sensitivity.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(12.0, 4.0))
for j, metric_name in enumerate(["L1_density_error", "truncated_KDE_chi2"]):
    for m in methods:
        sub = density_sensitivity_df[density_sensitivity_df.method == m].groupby("bandwidth_factor")[metric_name].mean().reset_index()
        ax[j].plot(sub["bandwidth_factor"], sub[metric_name], marker=method_marker(m), color=method_color(m), label=m)
    ax[j].set_yscale("log")
    ax[j].set_xlabel("bandwidth factor")
    ax[j].set_ylabel(metric_name.replace("_", " "))
    ax[j].set_title(metric_name.replace("_", " "))
ax[0].legend(fontsize=7, ncol=2)
savefig(fig, FIGDIR, "fig06d_bandwidth_sensitivity")
plt.show()

def zero_crossings(x, y):
    x = np.asarray(x); y = np.asarray(y)
    ids = np.where(np.signbit(y[:-1]) != np.signbit(y[1:]))[0]
    out = []
    for i in ids:
        denom = y[i+1] - y[i]
        out.append(float(x[i] - y[i]*(x[i+1]-x[i])/denom) if denom != 0 else float(x[i]))
    return out

drift_lo, drift_hi, drift_omitted = central_interval_from_cdf(target_grid, cdf_grid, mass=0.99)
drift_rows = []
fig, axes = plt.subplots(3, 2, figsize=(12.0, 8.2), sharex=True)
for j, m in enumerate(["LSC-adjacent", "LSC-overlong"]):
    drift_grid = score_cache[m][0]
    score_vals = score_cache[m][1]
    jump = jump_adjacent if m == "LSC-adjacent" else jump_overlong
    local_drift = eps * grad_logp(drift_grid)
    total_drift = local_drift + score_vals
    assert np.allclose(total_drift, eps * grad_logp(drift_grid) + score_vals)
    for xx, b0, ss, bt in zip(drift_grid, local_drift, score_vals, total_drift):
        drift_rows.append(dict(experiment="triple-well", method=m, x=float(xx),
                               local_Langevin_drift=float(b0),
                               Levy_score_correction=float(ss),
                               total_LSC_drift=float(bt),
                               jump_shell_center_positive=float(jump.a),
                               jump_shell_center_negative=float(-jump.a)))
    series = [
        ("local Langevin drift $-V'(x)$", local_drift, "tab:blue", "o"),
        ("Levy-score correction $S_{\\nu,\\varepsilon}(x)$", score_vals, "tab:orange", "^"),
        ("total corrected drift", total_drift, "tab:green", "s"),
    ]
    for row, (label, vals, color, marker) in enumerate(series):
        ax = axes[row, j]
        ax.plot(drift_grid, vals, color=color, lw=1.7)
        ax.axhline(0.0, color="black", lw=0.8)
        for mm in modes:
            ax.axvline(mm, color="black", ls="--", lw=0.75, alpha=0.40)
        for bd in [-1.5, 1.5]:
            ax.axvline(bd, color="tab:red", ls=":", lw=0.85, alpha=0.60)
        for rr in [-jump.a, jump.a]:
            ax.axvline(rr, color="tab:gray", ls=":", lw=0.85, alpha=0.50)
        for xx in zero_crossings(drift_grid, vals):
            if drift_lo <= xx <= drift_hi:
                ax.plot(xx, 0.0, marker=marker, ms=4, color=color)
        ax.set_xlim(drift_lo, drift_hi)
        ax.set_ylabel(label)
        if row == 0:
            ax.set_title(m)
        if row == 2:
            ax.set_xlabel("x")
pd.DataFrame(drift_rows).to_csv(TABDIR / "threewell_drift_grid.csv", index=False)
pd.DataFrame([dict(
    experiment="triple-well",
    plotted_interval_left=float(drift_lo),
    plotted_interval_right=float(drift_hi),
    omitted_target_mass=float(drift_omitted),
    full_domain_left=float(target_grid[0]),
    full_domain_right=float(target_grid[-1]),
    presentation="separate signed panels with shared x-axis",
)]).to_csv(TABDIR / "threewell_drift_figure_metadata.csv", index=False)
fig.suptitle("Triple-well corrected drift components on central 99% target interval", y=1.02)
savefig(fig, FIGDIR, "fig06e_drift_fields")
plt.show()


## Diagnostic: richer metrics, mode coverage, and weak observable errors

This block adds benchmark metrics inspired by the reference sampling notebooks: W2, MMD, effective mode coverage, effective mode count, and weak errors for several test functions.  It only adds diagnostics; the form-gap and abscissa analysis above is unchanged.


In [ ]:
def w2_distance_1d_quantile(x, y, max_points=5000, rng=None):
    rng = np.random.default_rng(777) if rng is None else rng
    x = np.asarray(x); y = np.asarray(y)
    n = int(min(len(x), len(y), max_points))
    if len(x) > n:
        x = rng.choice(x, size=n, replace=False)
    if len(y) > n:
        y = rng.choice(y, size=n, replace=False)
    xs = np.sort(x); ys = np.sort(y)
    return float(np.sqrt(np.mean((xs - ys)**2)))

def mode_coverage_from_mass(mass, threshold=0.03):
    mass = np.asarray(mass, dtype=float)
    return float(np.mean(mass > threshold))

def effective_mode_count_from_mass(mass):
    mass = np.asarray(mass, dtype=float)
    mass = mass / max(mass.sum(), 1e-300)
    H = -np.sum(np.where(mass > 0, mass*np.log(mass), 0.0))
    return float(np.exp(H) / len(mass))

def rich_metrics_threewell(all_runs):
    weak_fns = {
        "x": lambda z: z,
        "x2": lambda z: z**2,
        "x3": lambda z: z**3,
        "middle_indicator": lambda z: (classify_modes_1d(z) == 1).astype(float),
        "left_indicator": lambda z: (classify_modes_1d(z) == 0).astype(float),
    }
    target_vals = {name: float(np.sum(fn(target_grid)*p_grid*dxw_grid)) for name, fn in weak_fns.items()}
    rows=[]
    for r in all_runs:
        for t, xs in zip(r["times"], r["samples"]):
            cl = classify_modes_1d(xs)
            mass = np.array([np.mean(cl == i) for i in range(3)])
            row = dict(method=r["method"], seed=r["seed"], time=float(t),
                       coverage=mode_coverage_from_mass(mass), effective_mode_count=effective_mode_count_from_mass(mass))
            row["W2"] = w2_distance_1d_quantile(xs, target_samples, rng=np.random.default_rng(make_rng_seed(GLOBAL_SEED, 701, r["seed"], int(1000*t))))
            row["MMD"] = mmd_rbf_1d(xs, target_samples, bandwidth=1.0, max_points=900 if PROFILE != "paper" else 1600,
                                     rng=np.random.default_rng(make_rng_seed(GLOBAL_SEED, 702, r["seed"], int(1000*t))))
            for name, fn in weak_fns.items():
                row[f"weak_{name}"] = abs(float(np.mean(fn(xs))) - target_vals[name])
            rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(TABDIR / "threewell_rich_metrics_weak_errors.csv", index=False)
    display(pd.DataFrame([target_vals], index=["target_expectation"]))
    return df

three_rich_metrics = rich_metrics_threewell(all_runs)

fig, ax = plt.subplots(2, 3, figsize=FIGSIZE_GRID, sharex=True)
ax = ax.ravel()
panels = [
    ("coverage", "mode coverage"),
    ("effective_mode_count", "effective mode count"),
    ("W2", "Wasserstein-2 quantile"),
    ("MMD", "RBF MMD"),
    ("weak_middle_indicator", "weak error: middle indicator"),
    ("weak_x2", "weak error: x^2"),
]
for j, (metric, title) in enumerate(panels):
    for m in methods:
        plot_metric_with_ci(ax[j], three_rich_metrics, m, metric, logy=(metric not in ["coverage", "effective_mode_count"]), lw=1.4)
    for m in ["Langevin", "LSC-adjacent", "LSC-overlong"]:
        if metric not in ["coverage", "effective_mode_count"]:
            add_rate_reference_lines(ax[j], three_rich_metrics, metric, m,
                                     form_rate=gap_main.get(m, np.nan),
                                     abscissa_rate=globals().get("abscissa_main", {}).get(m, np.nan),
                                     logy=True)
    ax[j].set_title(title)
    ax[j].set_xlabel("time")
    ax[j].legend(fontsize=6, ncol=2)
savefig(fig, FIGDIR, "fig04d_rich_metrics_mode_coverage_weak_errors")
plt.show()


## Diagnostic: observed exponential decay-rate fit

The form-gap and abscissa lines are visual rate references.  This cell fits a simple log-linear decay rate from empirical mean curves on a middle-time window.  The reported standard error is an ordinary regression diagnostic and does not include all Monte Carlo uncertainty, time correlation, or reference-sample noise.  The seed-level confidence bands above remain the primary uncertainty visualization.


In [ ]:
rate_fit_df = fit_observed_decay_rates(
    metrics, methods, ["mode_TV", "middle_error", "W1", "CDF_sup"],
    t_min_frac=0.25, t_max_frac=0.85
)
rate_fit_df.to_csv(TABDIR/"threewell_observed_decay_rate_fits.csv", index=False)
display(rate_fit_df)


seed_rate_summary_df, seed_rate_by_seed_df = fit_seed_level_decay_rates(
    metrics, methods, ["mode_TV", "middle_error", "W1", "CDF_sup"],
    t_min_frac=0.25, t_max_frac=0.85
)
seed_rate_summary_df.to_csv(TABDIR/"threewell_observed_decay_rate_seed_level_summary.csv", index=False)
seed_rate_by_seed_df.to_csv(TABDIR/"threewell_observed_decay_rate_seed_level_by_seed.csv", index=False)
print("Seed-level decay-rate summary.  Its SE is across fitted seed-level slopes; NaN when only one seed is available.")
display(seed_rate_summary_df)


## Diagnostic: compact rate comparison table

This table puts three different rate quantities side by side: the form-gap reference rate, the finite-dimensional generator-abscissa diagnostic, and the fitted observed decay rate from the selected Monte Carlo metric.  They are intentionally shown together, but they are not the same mathematical object.

The form gap is a conservative $$L^2(\mu)$$ reference rate for the diagonal form.  The abscissa is a finite-dimensional nonreversible-generator diagnostic with its own stationarity checks.  The observed fit is metric-dependent and finite-time; it is not a theorem-level bound.

Note: in the bar chart below, colors encode rate type rather than sampler identity; sampler identity is on the x-axis.  This avoids duplicating the method-color convention used in trajectory and density panels.

When seed-level fits are available, the reported observed-rate SE is computed across seed-level fitted slopes rather than from time-point OLS residuals.



In [ ]:

PRIMARY_RATE_METRIC = "middle_error"
rows = []
for m in methods:
    rate_source_df = seed_rate_summary_df if "seed_rate_summary_df" in globals() else rate_fit_df
    sub = rate_source_df[(rate_source_df.method == m) & (rate_source_df.metric == PRIMARY_RATE_METRIC)]
    rows.append(dict(
        method=m,
        primary_metric=PRIMARY_RATE_METRIC,
        form_gap=float(gap_main.get(m, np.nan)),
        generator_abscissa=float(abscissa_main.get(m, np.nan)) if 'abscissa_main' in globals() else np.nan,
        r_obs=float(sub.r_obs.iloc[0]) if len(sub) else np.nan,
        r_obs_se=float(sub.r_obs_se.iloc[0]) if len(sub) else np.nan,
        n_seed_fit=int(sub.n_seed_fit.iloc[0]) if len(sub) and "n_seed_fit" in sub.columns and np.isfinite(sub.n_seed_fit.iloc[0]) else np.nan,
        fit_t_min=float(sub.t_min.iloc[0]) if len(sub) and 't_min' in sub.columns else np.nan,
        fit_t_max=float(sub.t_max.iloc[0]) if len(sub) and 't_max' in sub.columns else np.nan,
    ))
rate_comparison_df = pd.DataFrame(rows)
rate_comparison_df.to_csv(TABDIR / "threewell_rate_comparison.csv", index=False)
display(rate_comparison_df)

fig, ax = plt.subplots(1, 1, figsize=FIGSIZE_ROW)
x = np.arange(len(rate_comparison_df))
w = 0.26
ax.bar(x - w, rate_comparison_df.form_gap, width=w, label="form gap")
ax.bar(x, rate_comparison_df.generator_abscissa, width=w, label="abscissa")
ax.bar(x + w, rate_comparison_df.r_obs, width=w, yerr=rate_comparison_df.r_obs_se, capsize=3, label=f"observed fit: {PRIMARY_RATE_METRIC}")
ax.set_xticks(x); ax.set_xticklabels(rate_comparison_df.method, rotation=20)
ax.set_ylabel("rate")
ax.set_title("form gap, generator abscissa, and fitted observed decay")
ax.legend(fontsize=8)
savefig(fig, FIGDIR, "fig03b_rate_comparison_table")
plt.show()


## Diagnostic: time-step sensitivity for observed decay

This cell checks whether the observed Monte Carlo decay is stable under a small sweep of the Euler time step.  It uses fewer particles and only the main diagnostic observables, so it is not a replacement for the main simulation; it is a discretization-bias sanity check.


In [ ]:
def run_dt_sensitivity_three():
    if PROFILE == "smoke":
        dt_scales = [1.0]
        n_particles = min(40, CFG["n_particles"])
        base_steps = min(20, CFG["n_steps"])
    else:
        dt_scales = [0.5, 1.0, 2.0]
        n_particles = min(700, CFG["n_particles"])
        base_steps = min(700, CFG["n_steps"])
    rows=[]
    rng=np.random.default_rng(GLOBAL_SEED+1909)
    x0_base=make_initial_cloud_1d(rng, center=-3.0, scale=0.15, n=n_particles, clip=x_clip)
    for scale in dt_scales:
        dt=float(CFG["dt"]*scale)
        n_steps=max(4, int(round(base_steps/scale)))
        rec=max(1, int(round(CFG["record_every"]/scale)))
        for m in ["Langevin", "LSC-adjacent", "LSC-overlong"]:
            rr=simulate_1d_three(m, make_rng_seed(GLOBAL_SEED, 1909, int(scale*1000), methods.index(m)),
                                 x0_base.copy(), seed_id=f"dt{scale}", score_cache=score_cache,
                                 dt_override=dt, n_steps_override=n_steps, rec_override=rec)
            for t,xs in zip(rr["times"],rr["samples"]):
                modes_emp=classify_modes_1d(xs)
                mass=np.array([np.mean(modes_emp==i) for i in range(3)])
                rows.append(dict(method=m, dt=dt, dt_scale=scale, time=t,
                                 middle_error=abs(mass[1]-target_mode_mass[1]),
                                 mode_TV=0.5*np.sum(np.abs(mass-target_mode_mass))))
    sens=pd.DataFrame(rows)
    sens.to_csv(TABDIR/"threewell_dt_sensitivity.csv", index=False)
    fig,ax=plt.subplots(1,1,figsize=FIGSIZE_ROW)
    for (m,scale),sub in sens.groupby(["method","dt_scale"]):
        c=method_color(m)
        ls="-" if scale==1.0 else ("--" if scale<1 else ":")
        ax.semilogy(sub.time, np.maximum(sub.middle_error,1e-12), ls=ls, color=c, lw=1.7, label=f"{m}, dt x{scale}")
    ax.set_title("dt sensitivity: middle-well occupancy error")
    ax.set_xlabel("time"); ax.set_ylabel("middle error")
    ax.legend(fontsize=7, ncol=2)
    savefig(fig, FIGDIR, "fig04c_dt_sensitivity")
    plt.show()
    display(sens.groupby(["method","dt_scale"]).tail(1))
    return sens

dt_sensitivity_df = run_dt_sensitivity_three()


## 4b. Particle trajectory visualization

The three-well case is where trajectories are especially informative: overlong jumps can move left-to-right while under-visiting the middle well.  The panels below show fixed particle indices from a representative seed.


In [ ]:

# Particle trajectory visualization: fixed particle indices, one representative seed.
rep_seed = sorted(metrics["seed"].unique())[0]
n_traj = 26 if PROFILE != "smoke" else 10
fig, axes = plt.subplots(2, 3, figsize=(15.2, 7.8), sharex=True, sharey=True)
axes = axes.ravel()
for ax, m in zip(axes, methods):
    run0 = next(r for r in all_runs if r["method"] == m and r["seed"] == rep_seed)
    ts = run0["times"]
    Xtraj = np.vstack(run0["samples"])
    ids = np.linspace(0, Xtraj.shape[1]-1, min(n_traj, Xtraj.shape[1]), dtype=int)
    for pid in ids:
        ax.plot(ts, Xtraj[:, pid], lw=0.85, alpha=0.42)
    for mm in modes:
        ax.axhline(mm, color="k", ls="--", lw=0.75, alpha=0.40)
    ax.axhline(-1.5, color="tab:red", ls=":", lw=0.9, alpha=0.65)
    ax.axhline( 1.5, color="tab:red", ls=":", lw=0.9, alpha=0.65)
    ax.set_title(f"{m}: sample trajectories")
    ax.set_xlabel("time"); ax.set_ylabel("x(t)")
    ax.set_xlim(metrics.time.min(), metrics.time.max())
    ax.set_ylim(xlim_pdf)
axes[-1].axis("off")
savefig(fig, FIGDIR, "fig04b_particle_trajectories")
plt.show()


## 5. Ablations: radius and width expose the missed-middle mechanism

In [ ]:
if PROFILE=="smoke":
    a_grid=[3.0,6.0]
    h_grid=[0.18]
else:
    a_grid=[2.4,2.7,3.0,3.3,4.2,5.2,6.0,6.6]
    h_grid=[0.08,0.14,0.18,0.28,0.42]
Nabl=CFG["gap_Ns"][-1] if PROFILE!="paper" else CFG["gap_Ns"][-2]
rows=[]
for a0 in a_grid:
    j=ShellJump1D(a=a0,h=0.18,lam=1.2,quad_r=CFG["quad_r"],name=f"a={a0}")
    gap,diag,*_=full_gap_1d_form(logp,eps,CFG["gap_interval"][0],CFG["gap_interval"][1],N=Nabl,jump=j)
    rows.append(dict(kind="radius",value=a0,gap=gap))
for h0 in h_grid:
    for a0,label in [(3.0,"adjacent"),(6.0,"overlong")]:
        j=ShellJump1D(a=a0,h=h0,lam=1.2,quad_r=CFG["quad_r"],name=f"{label},h={h0}")
        gap,diag,*_=full_gap_1d_form(logp,eps,CFG["gap_interval"][0],CFG["gap_interval"][1],N=Nabl,jump=j)
        rows.append(dict(kind=f"width-{label}",value=h0,gap=gap))
abl=pd.DataFrame(rows)
abl.to_csv(TABDIR/"threewell_gap_ablations.csv",index=False)
display(abl)

fig,ax=plt.subplots(1,2,figsize=(11,4))
sub=abl[abl.kind=="radius"].sort_values("value")
ax[0].plot(sub.value,sub.gap,marker="o",lw=2)
ax[0].axvline(3,color="tab:green",ls="--",label="adjacent")
ax[0].axvline(6,color="tab:red",ls="--",label="overlong")
ax[0].set_title("gap vs jump radius")
ax[0].set_xlabel("shell center a"); ax[0].set_ylabel(r"$\lambda_{\rm full}$")
ax[0].legend()
for kind in ["width-adjacent","width-overlong"]:
    sub=abl[abl.kind==kind].sort_values("value")
    ax[1].plot(sub.value,sub.gap,marker="o",lw=2,label=kind)
ax[1].set_title("gap vs shell width h")
ax[1].set_xlabel("h"); ax[1].set_ylabel(r"$\lambda_{\rm full}$")
ax[1].legend()
savefig(fig, FIGDIR, "fig05_gap_ablations")
plt.show()


## Diagnostic: jump-event geometry versus continuous trajectory interpolation

The particle trajectories connect recorded positions by straight lines, so a
line segment may visually suggest a small jump even when the actual compound
Poisson increment is large.  The following event-level diagnostic samples the
jump law directly and classifies where an ideal jump from each well center
would land.  This separates the jump mechanism from the continuous diffusion
and drift occurring between recorded times.


In [ ]:

def jump_event_diagnostics_1d(jump_map, centers, classifier, label_prefix=''):
    rng = np.random.default_rng(GLOBAL_SEED + 8844)
    fig, ax = plt.subplots(len(jump_map), 3, figsize=(14.2, 3.8*len(jump_map)))
    if len(jump_map) == 1:
        ax = np.array([ax])
    trans_rows=[]
    for row, (name, jump) in enumerate(jump_map.items()):
        inc = jump.sample_jumps(rng, 70000 if PROFILE != 'smoke' else 6000)
        ax[row,0].hist(inc, bins=90, density=True, histtype='step', lw=1.8, label=name)
        ax[row,0].set_title(f'{name}: sampled jump increments')
        ax[row,0].set_xlabel('jump increment r'); ax[row,0].set_ylabel('density'); ax[row,0].legend(fontsize=8)
        trans = np.zeros((len(centers), len(centers)))
        for i, c in enumerate(centers):
            y = c + inc
            labs = classifier(y)
            for j in range(len(centers)):
                trans[i,j] = np.mean(labs == j)
                trans_rows.append(dict(jump=name, source=i, target=j, prob=trans[i,j]))
        im = ax[row,1].imshow(trans, vmin=0, vmax=1)
        ax[row,1].set_title(f'{name}: jump-induced mode transition')
        ax[row,1].set_xlabel('landing mode'); ax[row,1].set_ylabel('source mode')
        for i in range(trans.shape[0]):
            for j in range(trans.shape[1]):
                ax[row,1].text(j, i, f'{trans[i,j]:.2f}', ha='center', va='center', fontsize=8)
        for i, c in enumerate(centers):
            yi = c + inc[:1200 if PROFILE!='smoke' else 250]
            ax[row,2].scatter(np.full_like(yi, c), yi, s=4, alpha=0.20, label=f'source {i}')
        for c in centers: ax[row,2].axhline(c, color='k', ls='--', lw=0.8, alpha=0.45)
        ax[row,2].set_title(f'{name}: endpoints from well centers')
        ax[row,2].set_xlabel('source center'); ax[row,2].set_ylabel('landing position')
        ax[row,2].legend(fontsize=7)
        fig.colorbar(im, ax=ax[row,1], fraction=0.046, pad=0.04)
    pd.DataFrame(trans_rows).to_csv(TABDIR / f'{label_prefix}jump_event_transition_probs.csv', index=False)
    try:
        fig.tight_layout()
    except Exception:
        pass
    savefig(fig, FIGDIR, f'{label_prefix}fig07_jump_event_diagnostics')
    plt.show()

jump_event_diagnostics_1d({'adjacent a=3': jump_adjacent, 'overlong a=6': jump_overlong},
                          modes, lambda y: classify_modes_1d(np.asarray(y)), label_prefix='threewell_')


## Output registry

In [ ]:
# Phase17C output registry and run summary.
from datetime import datetime
import json

table_files = sorted(str(p.relative_to(PROJECT_ROOT)) for p in (RELEASE_TABLE_DIR / "02_triple_well").glob("*.csv"))
figure_files = sorted(str(p.relative_to(PROJECT_ROOT)) for p in DIAGNOSTIC_FIG_DIR.glob(f"{FIG_PREFIX}*.pdf"))
registry = {
    "experiment": "1D triple-well",
    "table_directory": str((RELEASE_TABLE_DIR / "02_triple_well").relative_to(PROJECT_ROOT)),
    "figure_directory": str(DIAGNOSTIC_FIG_DIR.relative_to(PROJECT_ROOT)),
    "tables": table_files,
    "figures": figure_files,
    "created_utc_like": datetime.utcnow().isoformat(timespec="seconds") + "Z",
}
registry_path = RELEASE_LOG_DIR / "02_triple_well_output_registry.json"
registry_path.write_text(json.dumps(registry, indent=2) + "\n", encoding="utf-8")
pd.DataFrame([{"experiment": "1D triple-well", "n_tables": len(table_files), "n_figures": len(figure_files)}]).to_csv(
    RELEASE_TABLE_DIR / "02_triple_well_summary.csv", index=False
)
print("Phase17C registry written:", registry_path)
print("tables:", len(table_files), "figures:", len(figure_files))


In [ ]:
write_reference_registry(
    SPECTRAL_REFERENCE_RECORDS,
    TABDIR / "threewell_spectral_reference_lines.csv",
)
SCRIPTS_DIR = RELEASE_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from generate_canonical_release_figures import generate_triplewell_release
generate_triplewell_release()